 <center> <h1> <b> Pattern Recognition and Machine Learning (EE2802 - AI2000) </b> </h1> </center>

<b> Programming Assignment 02 - Regression : </b> Welcome to the programming assignment (PA) on regression. This programing assignment focuses on understanding the basic concepts of linear regression.


<b> Instructions </b>
1. Plagiarism is strictly prohibited.
2. Delayed submissions are not accepted
3. Please DO NOT use any machine learning libraries unless and otherwise specified.


<center> <h2> <b> Understanding Basic Concepts </b> </h2> </center>


<b> Part - (1) :  Understanding Error Surfaces </b>

According to www.geogebra.org, the relationship between human height (in
inches) and weight (in pounds) is given by <br>
<center> $t = 3.86x - 110.42$ </center>

(a) Generate 25 meaningful data points from this relationship, mimicking a
noisy sensor, where the noise follows a zero mean Gaussian with a variance
of 20. Plot the scatter plot of the data. <br>
(b) Now, we need to estimate the above relationship from the noisy data
generated in (a) by fitting a line, i.e $\hat{t} = y(x,w) = w_{0} + w_{1}x$. Let us use least squares criterion discussed in the class to estimate the parameters $w_{0}$ and $w_{1}$. Generate and plot the error surface $J(w_{0},w_{1})$ associated with this approach. Locate the minimum on this error surface.<br>

(c) Estimate the parameters using least squares approach, and compare them
with the desired values.
<center> $\textbf{w}_{opt} = (\textbf{X}^{T}\textbf{X})^{-1}\textbf{X}^{T}\textbf{t}$</center>

(d) Report all your observations

<b> Part - (2) : Understanding model order and overfitting  </b>

(a). Generate  20  data  points  from $t_{n} = sin(2πx_{n}) + e_{n}$, where $x_{n} \in [0,1] $ and $e_{n} \thicksim \mathcal{N} (0,0.1)$ , and divide them into two sets, a training set and a testing set, with each set containing 10 points <br>

(b). Fit  an $M^{th}$ degree  polynomial  to  the  training  data  using  least  squares approach, i.e.,
<center> $\hat{t_{n}} = w_{0} + w_{1}x + .... +  w_{m}x^{m} + ... + w_{M}x^{M} $ </center>

Use the estimated parameter vector $\textbf{w}$, to predict the target values in training and testing datasets.  Plot the root mean squared error associated with each dataset, for M=0,1,...,9. Explain your results. <br>

(c) Increase the size of the training dataset to 100 points, and repeat (b). <br>

(d) Add a $l_{2}$ regularization term to the objective function in (b) and repeat (b) and (c).  Study the affect of Lagrange multiplier λ on the root mean squared error of the training and testing datasets <br>

(e) Modify the function in (a) to $t_{n}=5+sin(2πx_{n})+e_{n}$ to study the effect of regularizing the bias coefficient $w_{0}$.

(f) Report all your observations

<b> Part - (3) : Understanding the choice of kernel  </b>


(a). Generate 100 data points from $t_{n}=sin(2πx_{n})+e_{n}$, where $x_{n} \in [0 1]$ and $e_{n} \thicksim \mathcal{N}(0,0.1)$, and divide them into two sets, a training set and a testing test each containing 50 points.  Fit an $M^{th}$ degree polynomial using polynomial,Gaussian and sigmoidal kernels, and study the goodness of fit in each case, for different model orders M

(b). Repeat (a) by modifying the target function to <br>
<center> $t_{n} = $ $\begin{cases}
 \text{sinusoid} + e_{n} , \;\; where \;\; x  \in [0,1) \\
 \text{triangle} + e_{n} , \;\; where \;\; x  \in [1,2) \\
 \text{Gaussian} + e_{n} , \;\; where \;\; x  \in [2,3) \\
\end{cases}$ </center>

Clearly discuss your observations/results for each of the three kernels.

(c). Report all your observations

<b> Part - (4) : Understanding online training  </b>

(a). Repeat 3(a) and 3(b) using stochastic gradient descent for weight update.Study the effect of step size η on convergence of the weights, and compare them to those obtained using closed form expressions in 3.  Plot the mse as a  function  of  iterations.

(b). Study the effect of batch size on the speed of convergence

(c). Report all your observations

<b> Part - (5) : Understanding bias-variance trade-off  </b>

(a). Generate L=100 datasets of noisy sinusoidal data, each having N=25  datapoints. For each dataset, fit a $M=25^{th}$ order linear regression model consisting of 24 Gaussian basis functions and one bias parameter.  Use regularized least squares, governed by the parameter λ, to estimate the parameters $\textbf{w}$. Illustrate the concept of bias and variance using these 100 different parameter fits.
1.   Chose three different regularization coefficeints (low,middle and high)
2. For every regularization coefficient, produce two plots: one displaying 100 estimated curves, and the other showing the mean of the estimated curves alongside the original function.
2. For three regularization coefficients, you should have a total of six plots, meaning two plots for each regularization.
3. Using the six plots above, describe the bias-variance trade-off.


(b). Report all your observations







<b> Part - (6) : Understanding
Maximum a Posteriori (MAP) estimate  </b>

(a). Generate 100 noisy data points of a sinusoid. Fit a $20^{th}$  order  linear regression  model  with  Gaussian  basis  functions. Starting from a standard normal prior, update the statistics of the posterior density of the parameters using Bayesian sequential updates.

(b). Sample a parameter vector from the posterior distribution, and obtain the curve fit for this realization. Repeat this for several times, and estimate the average of these curve fits, and compare it with the original sinusoid

(c). Use the posterior distribution of the parameters to evaluate the predictive distribution of target $p(t_{0}/x_{0},X,t)$, and plot it for different number of training data points, as discussed in the class.

(d). Report all your observations

# PART 1

In [ ]:
#Understanding Error Surface
#All imports
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import collections
import math

########################################
#Generate meaningfull data
########################################

np.random.seed(42)
input = np.random.uniform(50, 80, size=(25, 1))
noise = np.random.normal(0, math.sqrt(20), size=(25, 1))
target = 3.86 * input - 110.42 + noise


########################################
#Plot scatter plot of data
########################################

#Plot the scatter plot of the data
plt.scatter(input, target, color='pink')
plt.xlabel('input (height in inches)')
plt.ylabel('target (weight in pounds)')
plt.show()



########################################
#Weight estimation through error surface, i.e., empirically locate the minima of error surface
########################################
#Sample a bunch of w's around w_opt and compute the associated error
w_opt = np.array([-110.42, 3.86])


#Compute the error
def Error(w,t,x): #inputs : 1)weight 2)data i.e (t,x)
    y = w[0] + w[1] * x
    error = np.mean(np.square(t - y))
    return error

w0_vals = np.linspace(-110.42-45, -110.42+45, 50)
w1_vals = np.linspace(3.86-15, 3.86+15, 50)
W_0, W_1 = np.meshgrid(w0_vals, w1_vals)
Error_surface = np.empty_like(W_0)
for i in range(len(w0_vals)):
    for j in range(len(w1_vals)):
        Error_surface[i, j] = Error([w0_vals[i], w1_vals[j]], target, input)

#Plot 3D error surface and the corresponding contour plots
#Error surface is a function of w0 and w1

fig = plt.figure(figsize=(10, 8))

ax = plt.axes(projection='3d')
ax.plot_surface(W_0, W_1, Error_surface, cmap='winter', alpha=0.4)
ax.set_title('3d plit of error', fontsize=14)
ax.set_xlabel('w_0', fontsize=12)
ax.set_ylabel('w_1', fontsize=12)
ax.set_zlabel('Error_surface', fontsize=12)

plt.show()



#Locate the minima of the error surface

min_error = np.min(Error_surface)
min_error_index = np.where(Error_surface == min_error)
min_error_w0 = w0_vals[min_error_index[0][0]]
min_error_w1 = w1_vals[min_error_index[1][0]]
plot_w_opt = np.array([min_error_w0, min_error_w1])


########################################
#Least squares approach to estimate the weights
########################################
#Complete the below linear regression function
def LinearRegression(x,t) -> np.ndarray: #inputs : 1)input data i.e (x). 2)target i.e (t)
    #Compute the optimal weights
    w_opt = np.linalg.inv(x.T @ x) @ x.T @ t
    return w_opt


#Estimate optimal weight's using "LinearRegression" function

x_values = np.column_stack((np.ones(25), input))

w_opt = LinearRegression(x_values, target)

#Estimate the targets using the input x and the estimated weights

predictions = x_values @ w_opt

def Prediction(x, w_pred) -> float:
    return w_pred[1]*x + w_pred[0]

X_train = np.linspace(50, 80, 300)

#Plot the estimated line on top of the above scatter plot

plt.scatter(input, target, color='pink', label = "true data")
plt.plot(X_train, Prediction(X_train, w_opt), color='red', label = "best fit line")
plt.xlabel("height")
plt.ylabel("weight")
plt.title("linear regression using pseudo inverse")
plt.grid()
plt.legend()
plt.show()

plt.scatter(input, target, color='pink', label = "true data")
plt.plot(X_train, Prediction(X_train, plot_w_opt), color='red', label = "best fit line")
plt.xlabel("Height")
plt.ylabel("Weight")
plt.title("linear regression using 3D plot")
plt.grid()
plt.legend()
plt.show()


########################################
#Compare the estimated weight's using least squares approach with the error surface approach
########################################
print("Predicted weights from the pseudo inverse solution:")
print(np.reshape(w_opt, (2,)))
print("Predicted weights from the 3D plot:")
print(plot_w_opt)




<b> Report your observations </b>

1. The error surface approach yielded closer to true soulutions, since sampling was done from near the true solution.

2. Pseudo Inverse soution has higher slope due to outlier.

3. As the data size increases, the predicted solution gets closer to the true solution.



# PART 2

In [ ]:
#Understanding model order and overfitting
########################################
#Generate 20 data points
#####################################curr###

np.random.seed(42)
input = np.random.uniform(0, 1, (20, 1))
noise = np.random.normal(0, 0.1, (20, 1))
output = np.sin(2*(math.pi)*input) + noise    



#Obtain train and test splits
X_train = input[:10]
X_test = input[10:20]
Y_train = output[:10]
Y_test = output[10:20]




########################################
#Fitting Mth degree polynomial using least squares approach
########################################
#Complete the function
def PolynomialFit(X_train,Y_train,M,lamda): #(training data, trining targets, Model order, Regularization coefficient)
    #Transform the data using polynomial kernel
    kernel = np.ones(X_train.shape)
    temp = np.ones(X_train.shape)
    # kernel = np.hstack([kernel] + [temp * X_train for _ in range(M)])
    for _ in range(M):
        temp = temp * X_train
        kernel = np.hstack((kernel, temp))
    #Find Pseudo inverse solution
    w_opt = np.linalg.inv(kernel.T @ kernel + lamda*np.identity(kernel.shape[1])) @ (kernel.T) @ (Y_train)
    #return the weight vector
    return w_opt


#Complete the function
def PolynomialPred(w_est,X_train,X_test): #(weight,training data, testing data, training targets, testing targets)
    #Estimate the targets for both training and testing data
    M = w_est.shape[0] - 1

    kernelTrain = np.ones(X_train.shape)
    temp = np.ones(X_train.shape)
    for _ in range(M):
       temp = temp * X_train
       kernelTrain = np.hstack((kernelTrain, temp))
    TrainPredictions = kernelTrain @ w_est

    kernelTest = np.ones(X_test.shape)
    temp = np.ones(X_test.shape)
    for _ in range(M):
       temp = temp * X_test
       kernelTest = np.hstack((kernelTest, temp))
    TestPredictions = kernelTest @ w_est

    #Return training and testing predictions
    return TrainPredictions, TestPredictions

#Complete the function
def PolynomialPred_Error(w_est,X_train,X_test,Y_train,Y_test): #(weight,training data, testing data, training targets, testing targets)
    #Estimate the targets for both training and testing data
    TrainPredictions, TestPredictions = PolynomialPred(w_est, X_train, X_test)

    TrainError = np.mean(np.square(TrainPredictions-Y_train))
    TestError = np.mean(np.square(TestPredictions-Y_test))

    #Return training and testing error
    return TrainError, TestError

#Iterate through range of M values
M_range=list(range(10))
TrError = []
TeError = []

TrainPredictionsList = []
TestPredictionsList = []

for M in M_range:
    #Fit Mth order polynomial i.e estimate optimal w. Use the function "PolynomialFit"
    w_est = PolynomialFit(X_train, Y_train, M ,0)

    #Predict training and testing targets
    train_predictions, test_predictions = PolynomialPred(w_est, X_train, X_test)
    TrainPredictionsList.append(train_predictions)
    TestPredictionsList.append(test_predictions)

    #Predict errors on both training and testing data using estimated w. Use the function "PolynomialPred_Error"
    train_error, test_error = PolynomialPred_Error(w_est, X_train, X_test, Y_train, Y_test)


    #Store them for plotting
    TrError.append(train_error)
    TeError.append(test_error)

    


#Plot training and testing estimates alogwith the original targets
for i in range(len(M_range)):
    plt.scatter(X_train, Y_train, color = "royalblue", label = "training data (true)")
    plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
    plt.scatter(X_train, TrainPredictionsList[i], color = "red", label = "training data (predicted)")
    plt.scatter(X_test, TestPredictionsList[i], color = "yellow", label = "testing data (predicted)")
    plt.grid()
    plt.legend()
    plt.xlabel("x")
    plt.ylabel("prediction (p(x))")
    plt.xlim(0, 1)
    plt.ylim(-2, 2)
    plt.title(f"predictions (M = {M_range[i]})")
    plt.show()



#Plot training error vs polynomial order, and testing error vs polynomial order
plt.plot(M_range, TrError)
plt.xlabel("polynomial order")
plt.ylabel("errors")
plt.title("polynomial orders vs training errors")
plt.show()

plt.plot(M_range, TeError)
plt.xlabel("polynomial order")
plt.ylabel("errors")
plt.title("polynomial orders vs testing errors")
plt.show()


########################################
#Increase the size of training data set to 100 points and repeat the experiments
########################################
np.random.seed(42)
input = np.random.uniform(0, 1, (100, 1))
noise = np.random.normal(0, 0.1, (100, 1))
output = np.sin(2*(math.pi)*input) + noise

#Obtain train and test splits
X_train = input[:50]
X_test = input[50:100]
Y_train = output[:50]
Y_test = output[50:100]

#Iterate through range of M values
M_range=list(range(10))
TrError = []
TeError = []

TrainPredictionsList = []
TestPredictionsList = []

for M in M_range:
    #Fit Mth order polynomial i.e estimate optimal w. Use the function "PolynomialFit"
    w_est = PolynomialFit(X_train, Y_train, M ,0)

    #Predict training and testing targets
    train_predictions, test_predictions = PolynomialPred(w_est, X_train, X_test)
    TrainPredictionsList.append(train_predictions)
    TestPredictionsList.append(test_predictions)

    #Predict errors on both training and testing data using estimated w. Use the function "PolynomialPred_Error"
    train_error, test_error = PolynomialPred_Error(w_est, X_train, X_test, Y_train, Y_test)


    #Store them for plotting
    TrError.append(train_error)
    TeError.append(test_error)

    


#Plot training and testing estimates alogwith the original targets
for i in range(len(M_range)):
    plt.scatter(X_train, Y_train, color = "royalblue", label = "training data (true)")
    plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
    plt.scatter(X_train, TrainPredictionsList[i], color = "red", label = "training data (predicted)")
    plt.scatter(X_test, TestPredictionsList[i], color = "yellow", label = "testing data (predicted)")
    plt.grid()
    plt.legend()
    plt.xlabel("x")
    plt.ylabel("prediction (p(x))")
    plt.xlim(0, 1)
    plt.ylim(-2, 2)
    plt.title(f"predictions (M = {M_range[i]}, lambda = 0)")
    plt.show()



#Plot training error vs polynomial order, and testing error vs polynomial order
plt.plot(M_range, TrError)
plt.xlabel("polynomial order")
plt.ylabel("error")
plt.title("polynomial orders vs training errors")
plt.show()

plt.plot(M_range, TeError)
plt.xlabel("Polynomial Order")
plt.ylabel("error")
plt.title("polynomial orders vs testing errors")
plt.show()








########################################
#Effect of regularization
########################################
#Consider a set of lambda's. For example: lamdas = [0, 1e-7 , 1e-4, 1e-2, 1]
#Repeat the experiments, i.e., plot the prediction and error in predictions with respect to model order. Contrast these results with those obtained without regularization.

lamdas = [0, 1e-7 , 1e-4, 1e-2, 1]

np.random.seed(42)
input = np.random.uniform(0, 1, (20,1))
noise = np.random.normal(0, 0.1, (20,1))
output = np.sin(2*(math.pi)*input) + noise    



#Obtain train and test splits
X_train = input[:10]
X_test = input[10:20]
Y_train = output[:10]
Y_test = output[10:20]


for lamda in lamdas:
    #Iterate through range of M values
    M_range=list(range(10))
    TrError = []
    TeError = []

    TrainPredictionsList = []
    TestPredictionsList = []

    for M in M_range:
        #Fit Mth order polynomial i.e estimate optimal w. Use the function "PolynomialFit"
        w_est = PolynomialFit(X_train, Y_train, M ,lamda)

        #Predict training and testing targets
        train_predictions, test_predictions = PolynomialPred(w_est, X_train, X_test)
        TrainPredictionsList.append(train_predictions)
        TestPredictionsList.append(test_predictions)

        #Predict errors on both training and testing data using estimated w. Use the function "PolynomialPred_Error"
        train_error, test_error = PolynomialPred_Error(w_est, X_train, X_test, Y_train, Y_test)


        #Store them for plotting
        TrError.append(train_error)
        TeError.append(test_error)

        


    #Plot training and testing estimates alogwith the original targets
    for i in range(len(M_range)):
        plt.scatter(X_train, Y_train, color = "royalblue", label = "training data (true)")
        plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
        plt.scatter(X_train, TrainPredictionsList[i], color = "red", label = "training data (predicted)")
        plt.scatter(X_test, TestPredictionsList[i], color = "yellow", label = "testing data (predicted)")
        plt.grid()
        plt.legend()
        plt.xlabel("x")
        plt.ylabel("prediction (p(x))")
        plt.xlim(0, 1)
        plt.ylim(-2, 2)
        plt.title(f"predictions (M = {M_range[i]}, lambda = {lamda})")
        plt.show()



    #Plot training error vs polynomial order, and testing error vs polynomial order
    plt.plot(M_range, TrError)
    plt.xlabel("polynomial order")
    plt.ylabel("error")
    plt.title("polynomial orders vs training errors")
    plt.show()

    plt.plot(M_range, TeError)
    plt.xlabel("Polynomial order")
    plt.ylabel("error")
    plt.title("polynomial orders vs testing errors")
    plt.show()






########################################
#Effect of bias regularization
########################################
#Modify the function i.e include bias

bias = 5

#Generate data

lamdas = [0, 1e-7 , 1e-4, 1e-2, 1]

np.random.seed(42)
input = np.random.uniform(0, 1, (20, 1))
noise = np.random.normal(0, 0.1, (20, 1))
output = np.sin(2*(math.pi)*input) + noise + bias

#Obtain train and test splits
X_train = input[:10]
X_test = input[10:20]
Y_train = output[:10]
Y_test = output[10:20]

#plot biased data
plt.scatter(input, output)
plt.xlabel('input')
plt.ylabel('output')
plt.show()



#Estimate the polynomial with and without regularization constraint
# without is include in lambda = 0
# for lamda in lamdas:
    #Iterate through range of M values
M_range=list(range(10))
TrError = []
TeError = []

TrainPredictionsList = []
TestPredictionsList = []

for M in M_range:
    #Fit Mth order polynomial i.e estimate optimal w. Use the function "PolynomialFit"
    w_est = PolynomialFit(X_train, Y_train, M ,0)

    #Predict training and testing targets
    train_predictions, test_predictions = PolynomialPred(w_est, X_train, X_test)
    TrainPredictionsList.append(train_predictions)
    TestPredictionsList.append(test_predictions)

    #Predict errors on both training and testing data using estimated w. Use the function "PolynomialPred_Error"
    train_error, test_error = PolynomialPred_Error(w_est, X_train, X_test, Y_train, Y_test)


    #Store them for plotting
    TrError.append(train_error)
    TeError.append(test_error)

    


#Plot training and testing estimates alogwith the original targets
for i in range(len(M_range)):
    plt.scatter(X_train, Y_train, color = "royalblue", label = "training data (true, biased)")
    plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true, biased)")
    plt.scatter(X_train, TrainPredictionsList[i], color = "red", label = "training data (predicted, biased)")
    plt.scatter(X_test, TestPredictionsList[i], color = "yellow", label = "testing data (predicted, biased)")
    plt.grid()
    plt.legend()
    plt.xlabel("x")
    plt.ylabel("prediction (p(x))")
    plt.xlim(0, 1)
    plt.ylim(3, 7)
    plt.title(f"predictions (M = {M_range[i]}, lambda = {lamda})")
    plt.show()



#Plot training error vs polynomial order, and testing error vs polynomial order
plt.plot(M_range, TrError)
plt.xlabel("polynomial order")
plt.ylabel("error")
plt.title("polynomial orders vs training errors")
plt.show()

plt.plot(M_range, TeError)
plt.xlabel("Polynomial Order")
plt.ylabel("error")
plt.title("poly orders vs testing errors")
plt.show()



#Compare the two estimated polynomials and report the observations



<b> Report your observations </b>

1. Having small dataset and large polynomial order (M) causes overfitting, and high testing error.

2. On regularization, the training error increases, but the testing error decreases, thus reducing overfitting.

3. Overfitting is prevented by having a larger dataset, even withouth regularization.

4. The data is underfitted for large values of lagrange multiplier, and thus has high training and testing errors.

5. Bias regularization does not affect the model, as the $w_{0}$ adjusts itself to the bias term.



# PART 3

### PART 3 a

In [ ]:
#Understanding the choice of kernel
########################################
#Generate 100 data points
########################################

np.random.seed(42)
input = np.random.uniform(0, 1, (100, 1))
noise = np.random.normal(0, 0.1, (100, 1))
output = np.sin(2*(math.pi)*input) + noise




#Obtian train and test splits
#Take even samples for training and odd samples for testing
X_train = np.array([input[i] for i in range(1, 100, 2)])
X_test = np.array([input[i] for i in range(0, 100, 2)])
Y_train = np.array([output[i] for i in range(1, 100, 2)])
Y_test = np.array([output[i] for i in range(0, 100, 2)])

def gaussian_kernel(x, mean, variance = 0.1):
    return np.exp(-1*(x-mean)**2/(2 * variance))

def sigmoid_kernel(x, mean, standard_deviation = 0.1):
  return 1/(1+np.exp(-(x-mean)/standard_deviation))

#Function to estimate the parameters
def KernelRegressionFit(X_train,Y_train,kernelType,M,lamda): #(training data, training targets, type of kernel, regularization coefficient)
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    #print(X_train.shape)
    #X_train = np.reshape(X_train, (1,-1))
    #Y_train = np.reshape(Y_train, (1,-1))
    X = []
    if kernelType=='polynomial':
        #Use polynomial kernel to transform the data
        X = np.ones(X_train.shape)
        curr = np.ones(X_train.shape)

        for _ in range(M):
            curr = curr * X_train
            X = np.hstack((X, curr))

    if kernelType=='gaussian':
        #Use Gaussian kernel to transform the data
        X = np.ones(X_train.shape)
        means = np.linspace(0, 1, M)
        ins, mean_list = np.meshgrid(X_train, means)
        temp2 = gaussian_kernel(ins, mean_list)
        X = np.hstack((X, temp2.T))

    if kernelType=='sigmoidal':
        #Use Sigmoid kernel to transform the data
        X = np.ones(X_train.shape)
        means = np.linspace(0, 1, M)
        ins, mean_list = np.meshgrid(X_train, means)
        temp2 = sigmoid_kernel(ins, mean_list)
        X = np.hstack((X, temp2.T))

    #Estimate weights using Pseudo iverse solution
    w_opt = np.linalg.inv(X.T @ X + lamda*np.identity(X.shape[1])) @ (X.T) @ Y_train

    #Return the estimated weights
    return w_opt

#Function to compute the training and testing errors from the current weight estimates
def KernelRegressionPred_Error(w_est,X_train,Y_train,X_test,Y_test,kernelType):
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    X_tr = []
    X_te = []
    #X_train = np.reshape(X_train, (1,-1))
    #Y_train = np.reshape(Y_train, (1,-1))
    #X_test = np.reshape(X_test, (1,-1))
    #Y_test = np.reshape(Y_test, (1,-1))
    M = len(w_est)-1
    if kernelType=='polynomial':
        #Use polynomial kernel to transform the data
        X_tr = np.ones(X_train.shape)
        curr = np.ones(X_train.shape)

        for _ in range(M):
            curr = curr * X_train
            X_tr = np.hstack((X_tr, curr))

        X_te = np.ones(X_test.shape)
        curr = np.ones(X_test.shape)

        for _ in range(M):
            curr = curr * X_test
            X_te = np.hstack((X_te, curr))

    if kernelType=='gaussian':
        #Use Gaussian kernel to transform the data
        means = np.linspace(0, 1, M)

        X_tr = np.ones(X_train.shape)
        ins, mean_list = np.meshgrid(X_train, means)
        temp = gaussian_kernel(ins, mean_list)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones(X_test.shape)
        ins, mean_list = np.meshgrid(X_test, means)
        temp = gaussian_kernel(ins, mean_list)
        X_te = np.hstack((X_te, temp.T))

    if kernelType=='sigmoidal':
        #Use Sigmoid kernel to transform the data
        means = np.linspace(0, 1, M)
        
        X_tr = np.ones(X_train.shape)
        ins, mean_list = np.meshgrid(X_train, means)
        temp = sigmoid_kernel(ins, mean_list)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones(X_test.shape)
        ins, mean_list = np.meshgrid(X_test, means)
        temp = sigmoid_kernel(ins, mean_list)
        X_te = np.hstack((X_te, temp.T))


    #Estimate training and testing targets
    train_predictions = X_tr @ w_est
    test_predictions = X_te @ w_est

    #Compute and return the training and testing errors
    train_error = np.mean(np.square(train_predictions-Y_train))
    test_error = np.mean(np.square(test_predictions-Y_test))

    return train_predictions, test_predictions, train_error, test_error


#Iterate through range of M values
M_range=list(range(10))

polynomial_tr_error = []
polynomial_te_error = []
gaussian_tr_error = []
gaussian_te_error = []
sigmoid_tr_error = []
sigmoid_te_error = []

polynomial_train_predictions_list = []
polynomial_test_predictions_list = []
gaussian_train_predictions_list = []
gaussian_test_predictions_list = []
sigmoid_train_predictions_list = []
sigmoid_test_predictions_list = []

for M in M_range:
    #Fit Mth order polynomial using three kernels i.e {Polynomial,Gaussian,Sigmoid}
    w_polynomial = KernelRegressionFit(X_train, Y_train, 'polynomial', M, 0)
    polynomial_train_predictions, polynomial_test_predictions, polynomial_train_error, polynomial_test_error = KernelRegressionPred_Error(w_polynomial, X_train, Y_train, X_test, Y_test, 'polynomial')
    polynomial_tr_error.append(polynomial_train_error)
    polynomial_te_error.append(polynomial_test_error)
    polynomial_train_predictions_list.append(polynomial_train_predictions)
    polynomial_test_predictions_list.append(polynomial_test_predictions)


    w_gaussian = KernelRegressionFit(X_train, Y_train, 'gaussian', M, 0)
    gaussian_train_predictions, gaussian_test_predictions, gaussian_train_error, gaussian_test_error = KernelRegressionPred_Error(w_gaussian, X_train, Y_train, X_test, Y_test, 'gaussian')
    gaussian_tr_error.append(gaussian_train_error)
    gaussian_te_error.append(gaussian_test_error)
    gaussian_train_predictions_list.append(gaussian_train_predictions)
    gaussian_test_predictions_list.append(gaussian_test_predictions)


    w_sigmoid = KernelRegressionFit(X_train, Y_train, 'sigmoidal', M, 0)
    sigmoid_train_predictions, sigmoid_test_predictions, sigmoid_train_error, sigmoid_test_error = KernelRegressionPred_Error(w_sigmoid, X_train, Y_train, X_test, Y_test, 'sigmoidal')
    sigmoid_tr_error.append(sigmoid_train_error)
    sigmoid_te_error.append(sigmoid_test_error)
    sigmoid_train_predictions_list.append(sigmoid_train_predictions)
    sigmoid_test_predictions_list.append(sigmoid_test_predictions)

#Plot the predicted training and testing targets alongside the original targets for various model orders and all three different kernels.

for i in range(len(M_range)):
    plt.figure(figsize=(20, 5))
    plt.subplot(1, 3, 1)
    plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
    plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
    plt.scatter(X_train, polynomial_train_predictions_list[i], color = "red", label = "training data (predicted)")
    plt.scatter(X_test, polynomial_test_predictions_list[i], color = "yellow", label = "testing data (predicted)")
    plt.grid()
    plt.legend()
    plt.xlabel("x")
    plt.ylabel("prediction (p(x))")
    plt.xlim(0, 1)
    plt.ylim(-2, 2)
    plt.title(f"polynomial kernel (M = {M_range[i]})")

    plt.subplot(1, 3, 2)
    plt.scatter(X_train, Y_train, color = "blue", label = "True Training Data")
    plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
    plt.scatter(X_train, gaussian_train_predictions_list[i], color = "red", label = "training data (predicted)")
    plt.scatter(X_test, gaussian_test_predictions_list[i], color = "yellow", label = "testing data (predicted)")
    plt.grid()
    plt.legend()
    plt.xlabel("x")
    plt.ylabel("prediction (p(x))")
    plt.xlim(0, 1)
    plt.ylim(-2, 2)
    plt.title(f"gaussian kernel (M = {M_range[i]})")

    plt.subplot(1, 3, 3)
    plt.scatter(X_train, Y_train, color = "blue", label = "True Training Data")
    plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
    plt.scatter(X_train, sigmoid_train_predictions_list[i], color = "red", label = "training data (predicted)")
    plt.scatter(X_test, sigmoid_test_predictions_list[i], color = "yellow", label = "testing data (predicted)")
    plt.grid()
    plt.legend()
    plt.xlabel("x")
    plt.ylabel("prediction (p(x))")
    plt.xlim(0, 1)
    plt.ylim(-2, 2)
    plt.title(f"sigmoid kernel (M = {M_range[i]})")

    plt.suptitle(f"predictions (M = {M_range[i]})")
    plt.show()



#Plot training error vs polynomial order and testing error vs polynomial order for all the three different kernels

# region Plot
plt.figure(figsize=(20, 5))
plt.subplot(1, 2, 1)
plt.plot(M_range, polynomial_tr_error)
plt.grid()
plt.xlabel("polynomial order")
plt.ylabel("error")
plt.title("polynomial orders vs training errors")

plt.subplot(1, 2, 2)
plt.plot(M_range, polynomial_te_error)
plt.grid()
plt.xlabel("polynomial order")
plt.ylabel("error")
plt.title("polynomial orders vs testing errors")

plt.show()

plt.figure(figsize=(20, 5))
plt.subplot(1, 2, 1)
plt.plot(M_range, gaussian_tr_error)
plt.grid()
plt.xlabel("polynomial order")
plt.ylabel("error")
plt.title("polynomial orders vs training errors")

plt.subplot(1, 2, 2)
plt.plot(M_range, gaussian_te_error)
plt.grid()
plt.xlabel("polynomial order")
plt.ylabel("error")
plt.title("polynomial orders vs testing errors")

plt.show()

plt.figure(figsize=(20, 5))
plt.subplot(1, 2, 1)
plt.plot(M_range, sigmoid_tr_error)
plt.grid()
plt.xlabel("polynomial order")
plt.ylabel("error")
plt.title("polynomial orders vs training errors")

plt.subplot(1, 2, 2)
plt.plot(M_range, sigmoid_te_error)
plt.grid()
plt.xlabel("polynomial order")
plt.ylabel("error")
plt.title("polynomial orders vs testing errors")

plt.show()
# endregion




### PART 3 b

In [ ]:
########################################
#Repeat the experiments by changing target function
########################################

def target_function(x):
    x = np.asarray(x)
    
    sinusoid = np.sin(2 * np.pi * x)
    triangle = np.where(x < 1.5, 1 * (x - 1), 1 * (2 - x))  
    gaussian = np.exp(-((x - 2.5) ** 2) / 0.5)  
    
    t_n = np.where(x < 1, sinusoid, np.where(x < 2, triangle, gaussian))
    return t_n

# np.random.seed(42)
input = np.random.uniform(0, 3, (100, 1))
noise = np.random.normal(0, 0.1, (100, 1))
output = target_function(input) + noise

#Obtian train and test splits
#Take even samples for training and odd samples for testing
X_train = np.array([input[i] for i in range(1, 100, 2)])
X_test = np.array([input[i] for i in range(0, 100, 2)])
Y_train = np.array([output[i] for i in range(1, 100, 2)])
Y_test = np.array([output[i] for i in range(0, 100, 2)])

def gaussian_kernel(x, mean, variance = 0.1):
    return np.exp(-1*(x-mean)**2/(2 * variance))

def sigmoid_kernel(x, mean, standard_deviation = 0.1):
  return 1/(1+np.exp(-(x-mean)/standard_deviation))

#Function to estimate the parameters
def KernelRegressionFit(X_train,Y_train,kernelType,M,lamda): #(training data, training targets, type of kernel, regularization coefficient)
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    #print(X_train.shape)
    #X_train = np.reshape(X_train, (1,-1))
    #Y_train = np.reshape(Y_train, (1,-1))
    X = []
    if kernelType=='polynomial':
        #Use polynomial kernel to transform the data
        X = np.ones(X_train.shape)
        curr = np.ones(X_train.shape)

        for _ in range(M):
            curr = curr * X_train
            X = np.hstack((X, curr))

    if kernelType=='gaussian':
        #Use Gaussian kernel to transform the data
        X = np.ones(X_train.shape)
        means = np.linspace(0, 3, M)
        temp1, M = np.meshgrid(X_train, means)
        temp2 = gaussian_kernel(temp1, M)
        X = np.hstack((X, temp2.T))

    if kernelType=='sigmoidal':
        #Use Sigmoid kernel to transform the data
        X = np.ones(X_train.shape)
        means = np.linspace(0, 3, M)
        temp1, M = np.meshgrid(X_train, means)
        temp2 = sigmoid_kernel(temp1, M)
        X = np.hstack((X, temp2.T))

    #Estimate weights using Pseudo iverse solution
    w_opt = np.linalg.inv(X.T @ X + lamda*np.identity(X.shape[1])) @ (X.T) @ Y_train

    #Return the estimated weights
    return w_opt

#Function to compute the training and testing errors from the current weight estimates
def KernelRegressionPred_Error(w_est,X_train,Y_train,X_test,Y_test,kernelType):
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    X_tr = []
    X_te = []
    #X_train = np.reshape(X_train, (1,-1))
    #Y_train = np.reshape(Y_train, (1,-1))
    #X_test = np.reshape(X_test, (1,-1))
    #Y_test = np.reshape(Y_test, (1,-1))
    M = len(w_est)-1
    if kernelType=='polynomial':
        #Use polynomial kernel to transform the data
        X_tr = np.ones(X_train.shape)
        curr = np.ones(X_train.shape)

        for _ in range(M):
            curr = curr * X_train
            X_tr = np.hstack((X_tr, curr))

        X_te = np.ones(X_test.shape)
        curr = np.ones(X_test.shape)

        for _ in range(M):
            curr = curr * X_test
            X_te = np.hstack((X_te, curr))

    if kernelType=='gaussian':
        #Use Gaussian kernel to transform the data
        means = np.linspace(0, 3, M)

        X_tr = np.ones(X_train.shape)
        temp1, M = np.meshgrid(X_train, means)
        temp = gaussian_kernel(temp1, M)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones(X_test.shape)
        temp1, M = np.meshgrid(X_test, means)
        temp = gaussian_kernel(temp1, M)
        X_te = np.hstack((X_te, temp.T))

    if kernelType=='sigmoidal':
        #Use Sigmoid kernel to transform the data
        means = np.linspace(0, 3, M)
        
        X_tr = np.ones(X_train.shape)
        temp1, M = np.meshgrid(X_train, means)
        temp = sigmoid_kernel(temp1, M)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones(X_test.shape)
        temp1, M = np.meshgrid(X_test, means)
        temp = sigmoid_kernel(temp1, M)
        X_te = np.hstack((X_te, temp.T))


    #Estimate training and testing targets
    train_predictions = X_tr @ w_est
    test_predictions = X_te @ w_est

    #Compute and return the training and testing errors
    train_error = np.mean(np.square(train_predictions-Y_train))
    test_error = np.mean(np.square(test_predictions-Y_test))

    return train_predictions, test_predictions, train_error, test_error


#Iterate through range of M values
M_range=list(range(10))

polynomial_tr_error = []
polynomial_te_error = []
gaussian_tr_error = []
gaussian_te_error = []
sigmoid_tr_error = []
sigmoid_te_error = []

polynomial_train_predictions_list = []
polynomial_test_predictions_list = []
gaussian_train_predictions_list = []
gaussian_test_predictions_list = []
sigmoid_train_predictions_list = []
sigmoid_test_predictions_list = []

for M in M_range:
    #Fit Mth order polynomial using three kernels i.e {Polynomial,Gaussian,Sigmoid}
    w_polynomial = KernelRegressionFit(X_train, Y_train, 'polynomial', M, 0)
    polynomial_train_predictions, polynomial_test_predictions, polynomial_train_error, polynomial_test_error = KernelRegressionPred_Error(w_polynomial, X_train, Y_train, X_test, Y_test, 'polynomial')
    polynomial_tr_error.append(polynomial_train_error)
    polynomial_te_error.append(polynomial_test_error)
    polynomial_train_predictions_list.append(polynomial_train_predictions)
    polynomial_test_predictions_list.append(polynomial_test_predictions)


    w_gaussian = KernelRegressionFit(X_train, Y_train, 'gaussian', M, 0)
    gaussian_train_predictions, gaussian_test_predictions, gaussian_train_error, gaussian_test_error = KernelRegressionPred_Error(w_gaussian, X_train, Y_train, X_test, Y_test, 'gaussian')
    gaussian_tr_error.append(gaussian_train_error)
    gaussian_te_error.append(gaussian_test_error)
    gaussian_train_predictions_list.append(gaussian_train_predictions)
    gaussian_test_predictions_list.append(gaussian_test_predictions)



    w_sigmoid = KernelRegressionFit(X_train, Y_train, 'sigmoidal', M, 0)
    sigmoid_train_predictions, sigmoid_test_predictions, sigmoid_train_error, sigmoid_test_error = KernelRegressionPred_Error(w_sigmoid, X_train, Y_train, X_test, Y_test, 'sigmoidal')
    sigmoid_tr_error.append(sigmoid_train_error)
    sigmoid_te_error.append(sigmoid_test_error)
    sigmoid_train_predictions_list.append(sigmoid_train_predictions)
    sigmoid_test_predictions_list.append(sigmoid_test_predictions)

#Plot the predicted training and testing targets alongside the original targets for various model orders and all three different kernels.

for i in range(len(M_range)):
    # region Plot
    plt.figure(figsize=(20, 5))
    plt.subplot(1, 3, 1)
    plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
    plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
    plt.scatter(X_train, polynomial_train_predictions_list[i], color = "red", label = "training data (predicted)")
    plt.scatter(X_test, polynomial_test_predictions_list[i], color = "yellow", label = "testing data (predicted)")
    plt.grid()
    plt.legend()
    plt.xlabel("x")
    plt.ylabel("prediction (p(x))")
    plt.xlim(0, 3)
    plt.ylim(-2, 2)
    plt.title(f"polynomial kernel (M = {M_range[i]})")

    plt.subplot(1, 3, 2)
    plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
    plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
    plt.scatter(X_train, gaussian_train_predictions_list[i], color = "red", label = "training data (predicted)")
    plt.scatter(X_test, gaussian_test_predictions_list[i], color = "yellow", label = "testing data (predicted)")
    plt.grid()
    plt.legend()
    plt.xlabel("x")
    plt.ylabel("prediction (p(x))")
    plt.xlim(0, 3)
    plt.ylim(-2, 2)
    plt.title(f"gaussian kernel (M = {M_range[i]})")

    plt.subplot(1, 3, 3)
    plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
    plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
    plt.scatter(X_train, sigmoid_train_predictions_list[i], color = "red", label = "training data (predicted)")
    plt.scatter(X_test, sigmoid_test_predictions_list[i], color = "yellow", label = "testing data (predicted)")
    plt.grid()
    plt.legend()
    plt.xlabel("x")
    plt.ylabel("prediction (p(x))")
    plt.xlim(0, 3)
    plt.ylim(-2, 2)
    plt.title(f"sigmoid kernel (M = {M_range[i]})")

    plt.suptitle(f"predictions (M = {M_range[i]})")
    plt.show()
    # endregion
    
#Plot training error vs polynomial order and testing error vs polynomial order for all the three different kernels
# region Plot
plt.figure(figsize=(20, 5))
plt.subplot(1, 2, 1)
plt.plot(M_range, polynomial_tr_error)
plt.grid()
plt.xlabel("polynomial order")
plt.ylabel("error")
plt.title("polynomial orders vs training errors")

plt.subplot(1, 2, 2)
plt.plot(M_range, polynomial_te_error)
plt.grid()
plt.xlabel("polynomial order")
plt.ylabel("error")
plt.title("polynomial orders vs testing errors")

plt.suptitle("errors for polynomial kernel")
plt.show()

plt.figure(figsize=(20, 5))
plt.subplot(1, 2, 1)
plt.plot(M_range, gaussian_tr_error)
plt.grid()
plt.xlabel("polynomial order")
plt.ylabel("error")
plt.title("polynomial orders vs training errors")

plt.subplot(1, 2, 2)
plt.plot(M_range, gaussian_te_error)
plt.grid()
plt.xlabel("polynomial order")
plt.ylabel("error")
plt.title("polynomial orders vs testing errors")

plt.suptitle("errors for gaussian kernel")
plt.show()

plt.figure(figsize=(20, 5))
plt.subplot(1, 2, 1)
plt.plot(M_range, sigmoid_tr_error)
plt.grid()
plt.xlabel("polynomial order")
plt.ylabel("error")
plt.title("polynomial orders vs training errors")

plt.subplot(1, 2, 2)
plt.plot(M_range, sigmoid_te_error)
plt.grid()
plt.xlabel("polynomial order")
plt.ylabel("error")
plt.title("polynomial orders vs testing errors")

plt.suptitle("errors for sigmoid kernel")
plt.show()

# endregion


<b> Report your observations </b>

1. As M increases. the training and testing errors decrease due to increasing non linearity of the model, capturing the data better.

2. For the simple sine function all kernels perform similar due to the function having similar local and global properties. 

3. For the complex piecewise function, the gaussian and sigmoidal kernels perform better than the polynomial kernel, as they are local kernels and can capture the local properties of the function better.



# PART 4

### part 4 3a repeated

In [ ]:
def ErrorPred(w_est,X_train,Y_train,X_test,Y_test,kernelType): #(estimated weight, training data, training targets, testing data, testing targets, type of the kernel )
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    X_tr = []
    X_te = []
    M = len(w_est)-1
    if kernelType=='polynomial':
        X_tr = np.ones(X_train.shape)
        temp = np.ones(X_train.shape)

        for _ in range(M):
            temp = temp * X_train
            X_tr = np.hstack((X_tr, temp))

        X_te = np.ones(X_test.shape)
        temp = np.ones(X_test.shape)

        for _ in range(M):
            temp = temp * X_test
            X_te = np.hstack((X_te, temp))

    if kernelType=='gaussian':
        means = np.linspace(0, 1, M)

        X_tr = np.ones(X_train.shape)
        meshx, meshy = np.meshgrid(X_train, means)
        temp = gaussian_kernel(meshx, meshy, 0.1)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones(X_test.shape)
        meshx, meshy = np.meshgrid(X_test, means)
        temp = gaussian_kernel(meshx, meshy, 0.1)
        X_te = np.hstack((X_te, temp.T))

    if kernelType=='sigmoidal':
        means = np.linspace(0, 1, M)
        
        X_tr = np.ones(X_train.shape)
        meshx, meshy = np.meshgrid(X_train, means)
        temp = sigmoid_kernel(meshx, meshy)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones(X_test.shape)
        meshx, meshy = np.meshgrid(X_test, means)
        temp = sigmoid_kernel(meshx, meshy)
        X_te = np.hstack((X_te, temp.T))

    #Estimate training and testing targets
    X_tr = X_tr @ w_est
    X_te = X_te @ w_est

    #Compute and return the training and testing errors
    train_error = np.mean(np.square(X_tr-Y_train))
    test_error = np.mean(np.square(X_te-Y_test))

    return train_error, test_error


def OnlineTraining(X_train,Y_train,X_test, Y_test, kernelType,M,Epochs,BatchSize,stepSize): #(training data, training targets, testing data, testing targets, tupe of the kernel, order of the mode, Number of epochs, Batch size, Step size)
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    #Initialize the weights
    weights = np.zeros((M+1, 1))
    X_train = np.reshape(X_train, (X_train.shape[0], 1))
    Y_train = np.reshape(Y_train, (Y_train.shape[0], 1))
    X_test = np.reshape(X_test, (X_test.shape[0], 1))
    Y_test = np.reshape(Y_test, (Y_test.shape[0], 1))
    #Initialize the necessary variables
    tr_errors = []
    te_errors = []

    #Iterate through epochs

    epochs = range(Epochs)
    for epoch in epochs:
        # print(f"Epoch {epoch+1}")
        #Compute the train and test errors using the current weights
        tr_error, te_error = ErrorPred(weights, X_train, Y_train, X_test, Y_test, kernelType)
        #Store training and testing errors for plotting
        tr_errors.append(tr_error)
        te_errors.append(te_error)

        #Shuffle the data
        data = np.hstack((X_train, Y_train))
        np.random.shuffle(data)

        batches = np.ceil(X_train.shape[0]/BatchSize)
        batches = int(batches)

        #Iterate through the batches
        for batch in range(batches):
            gradient = np.zeros((M+1, 1))
            if batch < batches - 1:
                data_batch = data[batch*BatchSize:(batch+1)*BatchSize]
            else:
                data_batch = data[batch*BatchSize:]

            mean_gradient = np.zeros((M+1, 1))

            #Iterate through the data points of obtained batch
            for n in range(len(data_batch)):
                X_tr = []
                x = np.reshape(data_batch[n, 0], (1, 1))
                y = data_batch[n, 1]
                if kernelType=='polynomial':
                    X_tr = np.ones((x.shape[0], 1))
                    curr = np.ones((x.shape[0], 1))

                    for _ in range(M):
                        curr = curr * x
                        X_tr = np.hstack((X_tr, curr))

                if kernelType=='gaussian':
                    X_tr = np.ones((x.shape[0], 1))
                    means = np.linspace(0, 1, M)
                    X, mesh = np.meshgrid(x, means)
                    temp2 = gaussian_kernel(X, mesh, 0.1)
                    X_tr = np.hstack((X_tr, temp2.T))

                if kernelType=='sigmoidal':
                    X_tr = np.ones((x.shape[0], 1))
                    means = np.linspace(0, 1, M)
                    X, mesh = np.meshgrid(x, means)
                    temp2 = sigmoid_kernel(X, mesh)
                    X_tr = np.hstack((X_tr, temp2.T))

                #Compute the gradient of weight's
                #Compute the running mean of the weights gradients for the batch update
                gradient = gradient + (y - X_tr @ weights) * X_tr.T
                mean_gradient = mean_gradient + gradient/(n+1)

            #Update the weights using mean gradient, consider using reasonable stepSize
            weights = weights + stepSize * mean_gradient

    #Plot training and testing error across the epochs
    epoch_nos = range(1, Epochs+1)

    plt.figure(figsize=(15, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epoch_nos, tr_errors, label = "testing errors")
    plt.legend()
    plt.grid()
    plt.title("train errors vs epoch")

    plt.subplot(1, 2, 2)
    plt.plot(epoch_nos, te_errors, label = "testing errors")
    plt.legend()
    plt.grid()
    plt.title("testing errors vs epoch")

    plt.suptitle(f"gradient descent ({kernelType})")
    plt.show()

    #Return the estimated weights
    return weights

def OnlinePred(w_est,X_train,X_test,kernelType): #(estimated weights, training data, testing data, type of the kernel )
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    #Initialize the required variables
    X_tr = []
    X_te = []
    M = len(w_est)-1

    #Obtain kernel representations
    if kernelType=='polynomial':
        X_tr = np.ones((X_train.shape[0], 1))
        curr = np.ones((X_train.shape[0], 1))

        for _ in range(M):
            curr = curr * X_train
            X_tr = np.hstack((X_tr, curr))

        X_te = np.ones((X_test.shape[0], 1))
        curr = np.ones((X_test.shape[0], 1))

        for _ in range(M):
            curr = curr * X_test
            X_te = np.hstack((X_te, curr))

    if kernelType=='gaussian':
        means = np.linspace(0, 1, M)

        X_tr = np.ones((X_train.shape[0], 1))
        temp1, M = np.meshgrid(X_train, means)
        temp = gaussian_kernel(temp1, M, 0.1)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones((X_test.shape[0], 1))
        temp1, M = np.meshgrid(X_test, means)
        temp = gaussian_kernel(temp1, M, 0.1)
        X_te = np.hstack((X_te, temp.T))

    if kernelType=='sigmoidal':
        means = np.linspace(0, 1, M)
        
        X_tr = np.ones((X_train.shape[0], 1))
        temp1, M = np.meshgrid(X_train, means)
        temp = sigmoid_kernel(temp1, M)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones((X_test.shape[0], 1))
        temp1, M = np.meshgrid(X_test, means)
        temp = sigmoid_kernel(temp1, M)
        X_te = np.hstack((X_te, temp.T))


    #Estimate training and testing targets
    X_tr = X_tr @ w_est
    X_te = X_te @ w_est

    #Compute and return the training and testing target estimates
    return X_tr, X_te

def OnlinePred_Error(w_est,X_train,Y_train,X_test,Y_test,kernelType): #(estimated weights, training data, training targets, testing data, testing targets, type of the kernel )
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    #Initialize the required variables
    X_tr = []
    X_te = []
    # check this
    X_tr, X_te = OnlinePred(w_est, X_train, X_test, kernelType)
    
    #Compute and return the training and testing errors
    Y_tr_error = np.mean(np.square(X_tr-Y_train))
    Y_te_error = np.mean(np.square(X_te-Y_test))

    return Y_tr_error, Y_te_error


##################################################
#Repeat 3a with online training
##################################################

def part3aOnline():
    # region 3a
    np.random.seed(42)
    input = np.random.uniform(0, 1, (100, 1))
    noise = np.random.normal(0, 0.1, (100, 1))
    output = np.sin(2*(math.pi)*input) + noise

    X_train = np.array([input[i] for i in range(1, 100, 2)])
    X_test = np.array([input[i] for i in range(0, 100, 2)])
    Y_train = np.array([output[i] for i in range(1, 100, 2)])
    Y_test = np.array([output[i] for i in range(0, 100, 2)])

    # print(X_train.shape)
    # print((X_train.shape[0], 1))

    # print(X_test.shape)
    # print((X_test.shape[0], 1))


    M_range=list(range(10))

    polynomial_tr_error = []
    polynomial_te_error = []
    gaussian_tr_error = []
    gaussian_te_error = []
    sigmoid_tr_error = []
    sigmoid_te_error = []

    polynomial_train_predictions_list = []
    polynomial_test_predictions_list = []
    gaussian_train_predictions_list = []
    gaussian_test_predictions_list = []
    sigmoid_train_predictions_list = []
    sigmoid_test_predictions_list = []

    Epochs = 1000
    BatchSize = 20
    stepSize = 0.005



    for M in M_range:
        print(f"Model Order: {M}")
        #Fit Mth order polynomial using three kernels i.e {Polynomial,Gaussian,Sigmoid}
        w_polynomial = OnlineTraining(X_train, Y_train, X_test, Y_test, 'polynomial', M, Epochs, BatchSize, stepSize)
        polynomial_train_predictions, polynomial_test_predictions = OnlinePred(w_polynomial, X_train, X_test, 'polynomial')
        polynomial_train_error, polynomial_test_error = OnlinePred_Error(w_polynomial, X_train, Y_train, X_test, Y_test, 'polynomial')
        polynomial_tr_error.append(polynomial_train_error)
        polynomial_te_error.append(polynomial_test_error)
        polynomial_train_predictions_list.append(polynomial_train_predictions)
        polynomial_test_predictions_list.append(polynomial_test_predictions)

        w_gaussian = OnlineTraining(X_train, Y_train, X_test, Y_test, 'gaussian', M, Epochs, BatchSize, stepSize)
        gaussian_train_predictions, gaussian_test_predictions = OnlinePred(w_gaussian, X_train, X_test, 'gaussian')
        gaussian_train_error, gaussian_test_error = OnlinePred_Error(w_gaussian, X_train, Y_train, X_test, Y_test, 'gaussian')
        gaussian_tr_error.append(gaussian_train_error)
        gaussian_te_error.append(gaussian_test_error)
        gaussian_train_predictions_list.append(gaussian_train_predictions)
        gaussian_test_predictions_list.append(gaussian_test_predictions)

        w_sigmoid = OnlineTraining(X_train, Y_train, X_test, Y_test, 'sigmoidal', M, Epochs, BatchSize, stepSize)
        sigmoid_train_predictions, sigmoid_test_predictions = OnlinePred(w_sigmoid, X_train, X_test, 'sigmoidal')
        sigmoid_train_error, sigmoid_test_error = OnlinePred_Error(w_sigmoid, X_train, Y_train, X_test, Y_test, 'sigmoidal')
        sigmoid_tr_error.append(sigmoid_train_error)
        sigmoid_te_error.append(sigmoid_test_error)
        sigmoid_train_predictions_list.append(sigmoid_train_predictions)
        sigmoid_test_predictions_list.append(sigmoid_test_predictions)

        plt.figure(figsize=(20, 5))
        plt.subplot(1, 3, 1)
        plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
        plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
        plt.scatter(X_train, polynomial_train_predictions, color = "red", label = "training data (predicted)")
        plt.scatter(X_test, polynomial_test_predictions, color = "yellow", label = "testing data (predicted)")
        plt.grid()
        plt.legend()
        plt.xlabel("x")
        plt.ylabel("prediction (p(x))")
        plt.xlim(0, 1)
        plt.ylim(-2, 2)
        plt.title(f"polynomial kernel (M = {M})")

        plt.subplot(1, 3, 2)
        plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
        plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
        plt.scatter(X_train, gaussian_train_predictions, color = "red", label = "training data (predicted)")
        plt.scatter(X_test, gaussian_test_predictions, color = "yellow", label = "testing data (predicted)")
        plt.grid()
        plt.legend()
        plt.xlabel("x")
        plt.ylabel("prediction (p(x))")
        plt.xlim(0, 1)
        plt.ylim(-2, 2)
        plt.title(f"gaussian kernel (M = {M})")

        plt.subplot(1, 3, 3)
        plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
        plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
        plt.scatter(X_train, sigmoid_train_predictions, color = "red", label = "training data (predicted)")
        plt.scatter(X_test, sigmoid_test_predictions, color = "yellow", label = "testing data (predicted)")
        plt.grid()
        plt.legend()
        plt.xlabel("x")
        plt.ylabel("prediction (p(x))")
        plt.xlim(0, 1)
        plt.ylim(-2, 2)
        plt.title(f"sigmoid kernel (M = {M})")

        plt.show()

    #Plot the predicted training and testing targets alongside the original targets for various model orders and all three different kernels.


    #Plot training error vs polynomial order and testing error vs polynomial order for all the three different kernels

    # region Plot
    plt.figure(figsize=(20, 5))
    plt.subplot(1, 2, 1)
    plt.plot(M_range, polynomial_tr_error)
    plt.grid()
    plt.xlabel("polynomial order")
    plt.ylabel("error")
    plt.title("training errors")

    plt.subplot(1, 2, 2)
    plt.plot(M_range, polynomial_te_error)
    plt.grid()
    plt.xlabel("polynomial order")
    plt.ylabel("error")
    plt.title("testing errors")

    plt.suptitle("polynomial kernel errors")
    plt.show()

    plt.figure(figsize=(20, 5))
    plt.subplot(1, 2, 1)
    plt.plot(M_range, gaussian_tr_error)
    plt.grid()
    plt.xlabel("polynomial order")
    plt.ylabel("error")
    plt.title("training errors")

    plt.subplot(1, 2, 2)
    plt.plot(M_range, gaussian_te_error)
    plt.grid()
    plt.xlabel("polynomial order")
    plt.ylabel("error")
    plt.title("testing errors")

    plt.suptitle("gaussian kernel errors")
    plt.show()

    plt.figure(figsize=(20, 5))
    plt.subplot(1, 2, 1)
    plt.plot(M_range, sigmoid_tr_error)
    plt.grid()
    plt.xlabel("polynomial order")
    plt.ylabel("error")
    plt.title("training errors")

    plt.subplot(1, 2, 2)
    plt.plot(M_range, sigmoid_te_error)
    plt.grid()
    plt.xlabel("Polynomial Order")
    plt.ylabel("error")
    plt.title("testing errors")

    plt.suptitle("sigmoid kernel errors")
    plt.show()
    # endregion

    # endregion

part3aOnline()


### part 4 3b repeated

In [ ]:
def ErrorPred(w_est,X_train,Y_train,X_test,Y_test,kernelType): #(estimated weight, training data, training targets, testing data, testing targets, type of the kernel )
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    X_tr = []
    X_te = []
    M = len(w_est)-1
    if kernelType=='polynomial':
        X_tr = np.ones(X_train.shape)
        temp = np.ones(X_train.shape)

        for _ in range(M):
            temp = temp * X_train
            X_tr = np.hstack((X_tr, temp))

        X_te = np.ones(X_test.shape)
        temp = np.ones(X_test.shape)

        for _ in range(M):
            temp = temp * X_test
            X_te = np.hstack((X_te, temp))

    if kernelType=='gaussian':
        means = np.linspace(0, 3, M)

        X_tr = np.ones(X_train.shape)
        meshx, meshy = np.meshgrid(X_train, means)
        temp = gaussian_kernel(meshx, meshy)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones(X_test.shape)
        meshx, meshy = np.meshgrid(X_test, means)
        temp = gaussian_kernel(meshx, meshy)
        X_te = np.hstack((X_te, temp.T))

    if kernelType=='sigmoidal':
        means = np.linspace(0, 3, M)
        
        X_tr = np.ones(X_train.shape)
        meshx, meshy = np.meshgrid(X_train, means)
        temp = sigmoid_kernel(meshx, meshy)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones(X_test.shape)
        meshx, meshy = np.meshgrid(X_test, means)
        temp = sigmoid_kernel(meshx, meshy)
        X_te = np.hstack((X_te, temp.T))

    #Estimate training and testing targets
    X_tr = X_tr @ w_est
    X_te = X_te @ w_est

    #Compute and return the training and testing errors
    train_error = np.mean(np.square(X_tr-Y_train))
    test_error = np.mean(np.square(X_te-Y_test))

    return train_error, test_error


def OnlineTraining(X_train,Y_train,X_test, Y_test, kernelType,M,Epochs,BatchSize,stepSize): #(training data, training targets, testing data, testing targets, tupe of the kernel, order of the mode, Number of epochs, Batch size, Step size)
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    #Initialize the weights
    weights = np.zeros((M+1, 1))
    X_train = np.reshape(X_train, (X_train.shape[0], 1))
    Y_train = np.reshape(Y_train, (Y_train.shape[0], 1))
    X_test = np.reshape(X_test, (X_test.shape[0], 1))
    Y_test = np.reshape(Y_test, (Y_test.shape[0], 1))
    #Initialize the necessary variables
    tr_errors = []
    te_errors = []

    #Iterate through epochs

    epochs = range(Epochs)
    for epoch in epochs:
        # print(f"Epoch {epoch+1}")
        #Compute the train and test errors using the current weights
        tr_error, te_error = ErrorPred(weights, X_train, Y_train, X_test, Y_test, kernelType)
        #Store training and testing errors for plotting
        tr_errors.append(tr_error)
        te_errors.append(te_error)

        #Shuffle the data
        data = np.hstack((X_train, Y_train))
        np.random.shuffle(data)

        batches = np.ceil(X_train.shape[0]/BatchSize)
        batches = int(batches)

        #Iterate through the batches
        for batch in range(batches):
            #Initialize the necessary variables
            gradient = np.zeros((M+1, 1))
            #Get a batch of data
            if batch < batches - 1:
                data_batch = data[batch*BatchSize:(batch+1)*BatchSize]
            else:
                data_batch = data[batch*BatchSize:]

            #Iterate through the data points of obtained batch
            for n in range(len(data_batch)):
                #Obtain kernel representation
                X_tr = []
                x = np.reshape(data_batch[n, 0], (1, 1))
                y = data_batch[n, 1]
                if kernelType=='polynomial':
                    X_tr = np.ones((x.shape[0], 1))
                    curr = np.ones((x.shape[0], 1))

                    for _ in range(M):
                        curr = curr * x
                        X_tr = np.hstack((X_tr, curr))

                if kernelType=='gaussian':
                    X_tr = np.ones((x.shape[0], 1))
                    means = np.linspace(0, 3, M)
                    X, mesh = np.meshgrid(x, means)
                    temp2 = gaussian_kernel(X, mesh)
                    X_tr = np.hstack((X_tr, temp2.T))

                if kernelType=='sigmoidal':
                    X_tr = np.ones((x.shape[0], 1))
                    means = np.linspace(0, 3, M)
                    X, mesh = np.meshgrid(x, means)
                    temp2 = sigmoid_kernel(X, mesh)
                    X_tr = np.hstack((X_tr, temp2.T))

                #Compute the gradient of weight's
                #Compute the running mean of the weights gradients for the batch update
                gradient = gradient + (y - X_tr @ weights) * X_tr.T

            #Update the weights using mean gradient, consider using reasonable stepSize
            weights = weights + stepSize * gradient

    #Plot training and testing error across the epochs
    epoch_nos = range(1, Epochs+1)

    plt.figure(figsize=(15, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epoch_nos, tr_errors, label = "Training Errors")
    plt.legend(loc = 'upper right')
    plt.grid()
    plt.title("Training Errors vs Epoch")

    plt.subplot(1, 2, 2)
    plt.plot(epoch_nos, te_errors, label = "Testing Errors")
    plt.legend(loc = 'upper right')
    plt.grid()
    plt.title("Testing Errors vs Epoch")

    plt.suptitle("Gradient Descent for " + kernelType)
    plt.show()

    #Return the estimated weights
    return weights

def OnlinePred(w_est,X_train,X_test,kernelType): #(estimated weights, training data, testing data, type of the kernel )
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    #Initialize the required variables
    X_tr = []
    X_te = []
    M = len(w_est)-1

    #Obtain kernel representations
    if kernelType=='polynomial':
        X_tr = np.ones((X_train.shape[0], 1))
        curr = np.ones((X_train.shape[0], 1))

        for _ in range(M):
            curr = curr * X_train
            X_tr = np.hstack((X_tr, curr))

        X_te = np.ones((X_test.shape[0], 1))
        curr = np.ones((X_test.shape[0], 1))

        for _ in range(M):
            curr = curr * X_test
            X_te = np.hstack((X_te, curr))

    if kernelType=='gaussian':
        means = np.linspace(0, 3, M)

        X_tr = np.ones((X_train.shape[0], 1))
        temp1, M = np.meshgrid(X_train, means)
        temp = gaussian_kernel(temp1, M)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones((X_test.shape[0], 1))
        temp1, M = np.meshgrid(X_test, means)
        temp = gaussian_kernel(temp1, M)
        X_te = np.hstack((X_te, temp.T))

    if kernelType=='sigmoidal':
        means = np.linspace(0, 3, M)
        
        X_tr = np.ones((X_train.shape[0], 1))
        temp1, M = np.meshgrid(X_train, means)
        temp = sigmoid_kernel(temp1, M)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones((X_test.shape[0], 1))
        temp1, M = np.meshgrid(X_test, means)
        temp = sigmoid_kernel(temp1, M)
        X_te = np.hstack((X_te, temp.T))


    #Estimate training and testing targets
    X_tr = X_tr @ w_est
    X_te = X_te @ w_est

    #Compute and return the training and testing target estimates
    return X_tr, X_te

def OnlinePred_Error(w_est,X_train,Y_train,X_test,Y_test,kernelType): #(estimated weights, training data, training targets, testing data, testing targets, type of the kernel )
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    #Initialize the required variables
    X_tr = []
    X_te = []
    # check this
    X_tr, X_te = OnlinePred(w_est, X_train, X_test, kernelType)
    
    #Compute and return the training and testing errors
    Y_tr_error = np.mean(np.square(X_tr-Y_train))
    Y_te_error = np.mean(np.square(X_te-Y_test))

    return Y_tr_error, Y_te_error



##################################################
#Repeat 3b with online training
##################################################

def part3bOnline():


    # region 3b
    def target_function(x):
        x = np.asarray(x)
        
        sinusoid = np.sin(2 * np.pi * x)
        triangle = np.where(x < 1.5, 1 * (x - 1), 1 * (2 - x))  
        gaussian = np.exp(-((x - 2.5) ** 2) / 0.5)  
        
        t_n = np.where(x < 1, sinusoid, np.where(x < 2, triangle, gaussian))
        return t_n

    np.random.seed(42)
    input = np.random.uniform(0, 3, (100, 1))
    noise = np.random.normal(0, 0.1, (100, 1))
    output = target_function(input) + noise

    #Obtian train and test splits
    #Take even samples for training and odd samples for testing
    X_train = np.array([input[i] for i in range(1, 100, 2)])
    X_test = np.array([input[i] for i in range(0, 100, 2)])
    Y_train = np.array([output[i] for i in range(1, 100, 2)])
    Y_test = np.array([output[i] for i in range(0, 100, 2)])

    M_range=list(range(10))

    polynomial_tr_error = []
    polynomial_te_error = []
    gaussian_tr_error = []
    gaussian_te_error = []
    sigmoid_tr_error = []
    sigmoid_te_error = []

    Epochs = 250
    BatchSize = 20
    stepSize = 0.005

    for M in M_range:
        print(f"Model Order: {M}")
        #Fit Mth order polynomial using three kernels i.e {Polynomial,Gaussian,Sigmoid}
        w_polynomial = OnlineTraining(X_train, Y_train, X_test, Y_test, 'polynomial', M, Epochs, BatchSize, stepSize)
        polynomial_train_predictions, polynomial_test_predictions = OnlinePred(w_polynomial, X_train, X_test, 'polynomial')
        polynomial_train_error, polynomial_test_error = OnlinePred_Error(w_polynomial, X_train, Y_train, X_test, Y_test, 'polynomial')
        polynomial_tr_error.append(polynomial_train_error)
        polynomial_te_error.append(polynomial_test_error)
        polynomial_train_predictions_list.append(polynomial_train_predictions)
        polynomial_test_predictions_list.append(polynomial_test_predictions)


        w_gaussian = OnlineTraining(X_train, Y_train, X_test, Y_test, 'gaussian', M, Epochs, BatchSize, stepSize)
        gaussian_train_predictions, gaussian_test_predictions = OnlinePred(w_gaussian, X_train, X_test, 'gaussian')
        gaussian_train_error, gaussian_test_error = OnlinePred_Error(w_gaussian, X_train, Y_train, X_test, Y_test, 'gaussian')
        gaussian_tr_error.append(gaussian_train_error)
        gaussian_te_error.append(gaussian_test_error)
        gaussian_train_predictions_list.append(gaussian_train_predictions)
        gaussian_test_predictions_list.append(gaussian_test_predictions)

        w_sigmoid = OnlineTraining(X_train, Y_train, X_test, Y_test, 'sigmoidal', M, Epochs, BatchSize, stepSize)
        sigmoid_train_predictions, sigmoid_test_predictions = OnlinePred(w_sigmoid, X_train, X_test, 'sigmoidal')
        sigmoid_train_error, sigmoid_test_error = OnlinePred_Error(w_sigmoid, X_train, Y_train, X_test, Y_test, 'sigmoidal')
        sigmoid_tr_error.append(sigmoid_train_error)
        sigmoid_te_error.append(sigmoid_test_error)
        sigmoid_train_predictions_list.append(sigmoid_train_predictions)
        sigmoid_test_predictions_list.append(sigmoid_test_predictions)

        plt.figure(figsize=(20, 5))
        plt.subplot(1, 3, 1)
        plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
        plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
        plt.scatter(X_train, polynomial_train_predictions, color = "red", label = "training data (predicted)")
        plt.scatter(X_test, polynomial_test_predictions, color = "yellow", label = "testing data (predicted)")
        plt.grid()
        plt.legend()
        plt.xlabel("x")
        plt.ylabel("prediction (p(x))")
        plt.xlim(0, 3)
        plt.ylim(-2, 2)
        plt.title(f"polynomial kernel (M = {M})")

        plt.subplot(1, 3, 2)
        plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
        plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
        plt.scatter(X_train, gaussian_train_predictions, color = "red", label = "training data (predicted)")
        plt.scatter(X_test, gaussian_test_predictions, color = "yellow", label = "testing data (predicted)")
        plt.grid()
        plt.legend()
        plt.xlabel("x")
        plt.ylabel("prediction (p(x))")
        plt.xlim(0, 3)
        plt.ylim(-2, 2)
        plt.title(f"gaussian kernel (M = {M})")

        plt.subplot(1, 3, 3)
        plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
        plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
        plt.scatter(X_train, sigmoid_train_predictions, color = "red", label = "training data (predicted)")
        plt.scatter(X_test, sigmoid_test_predictions, color = "yellow", label = "testing data (predicted)")
        plt.grid()
        plt.legend()
        plt.xlabel("x")
        plt.ylabel("prediction (p(x))")
        plt.xlim(0, 3)
        plt.ylim(-2, 2)
        plt.title(f"sigmoid kernel (M = {M})")

        plt.show()

    # region Plot
    plt.figure(figsize=(20, 5))
    plt.subplot(1, 2, 1)
    plt.plot(M_range, polynomial_tr_error)
    plt.grid()
    plt.xlabel("polynomial order")
    plt.ylabel("error")
    plt.title("polynomial orders vs training errors")

    plt.subplot(1, 2, 2)
    plt.plot(M_range, polynomial_te_error)
    plt.grid()
    plt.xlabel("polynomial order")
    plt.ylabel("error")
    plt.title("polynomial orders vs testing errors")

    plt.suptitle("polynomial kernel errors")
    plt.show()

    plt.figure(figsize=(20, 5))
    plt.subplot(1, 2, 1)
    plt.plot(M_range, gaussian_tr_error)
    plt.grid()
    plt.xlabel("polynomial order")
    plt.ylabel("error")
    plt.title("polynomial orders vs training errors")

    plt.subplot(1, 2, 2)
    plt.plot(M_range, gaussian_te_error)
    plt.grid()
    plt.xlabel("polynomial order")
    plt.ylabel("error")
    plt.title("polynomial orders vs testing errors")

    plt.suptitle("gaussian kernel errors")
    plt.show()

    plt.figure(figsize=(20, 5))
    plt.subplot(1, 2, 1)
    plt.plot(M_range, sigmoid_tr_error)
    plt.grid()
    plt.xlabel("polynomial order")
    plt.ylabel("error")
    plt.title("polynomial orders vs training errors")

    plt.subplot(1, 2, 2)
    plt.plot(M_range, sigmoid_te_error)
    plt.grid()
    plt.xlabel("polynomial order")
    plt.ylabel("error")
    plt.title("polynomial orders vs testing errors")

    plt.suptitle("sigmoid kernel errors")
    plt.show()
    # endregion

    # endregion

part3bOnline()




### part 4 step size

In [ ]:
def ErrorPred(w_est,X_train,Y_train,X_test,Y_test,kernelType): #(estimated weight, training data, training targets, testing data, testing targets, type of the kernel )
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    X_tr = []
    X_te = []
    M = len(w_est)-1
    if kernelType=='polynomial':
        X_tr = np.ones(X_train.shape)
        temp = np.ones(X_train.shape)

        for _ in range(M):
            temp = temp * X_train
            X_tr = np.hstack((X_tr, temp))

        X_te = np.ones(X_test.shape)
        temp = np.ones(X_test.shape)

        for _ in range(M):
            temp = temp * X_test
            X_te = np.hstack((X_te, temp))

    if kernelType=='gaussian':
        means = np.linspace(0, 1, M)

        X_tr = np.ones(X_train.shape)
        meshx, meshy = np.meshgrid(X_train, means)
        temp = gaussian_kernel(meshx, meshy)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones(X_test.shape)
        meshx, meshy = np.meshgrid(X_test, means)
        temp = gaussian_kernel(meshx, meshy)
        X_te = np.hstack((X_te, temp.T))

    if kernelType=='sigmoidal':
        means = np.linspace(0, 1, M)
        
        X_tr = np.ones(X_train.shape)
        meshx, meshy = np.meshgrid(X_train, means)
        temp = sigmoid_kernel(meshx, meshy)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones(X_test.shape)
        meshx, meshy = np.meshgrid(X_test, means)
        temp = sigmoid_kernel(meshx, meshy)
        X_te = np.hstack((X_te, temp.T))

    #Estimate training and testing targets
    X_tr = X_tr @ w_est
    X_te = X_te @ w_est

    #Compute and return the training and testing errors
    train_error = np.mean(np.square(X_tr-Y_train))
    test_error = np.mean(np.square(X_te-Y_test))

    return train_error, test_error


def OnlineTraining(X_train,Y_train,X_test, Y_test, kernelType,M,Epochs,BatchSize,stepSize): #(training data, training targets, testing data, testing targets, tupe of the kernel, order of the mode, Number of epochs, Batch size, Step size)
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    #Initialize the weights
    weights = np.zeros((M+1, 1))
    X_train = np.reshape(X_train, (X_train.shape[0], 1))
    Y_train = np.reshape(Y_train, (Y_train.shape[0], 1))
    X_test = np.reshape(X_test, (X_test.shape[0], 1))
    Y_test = np.reshape(Y_test, (Y_test.shape[0], 1))
    #Initialize the necessary variables
    tr_errors = []
    te_errors = []

    #Iterate through epochs

    epochs = range(Epochs)
    for epoch in epochs:
        # print(f"Epoch {epoch+1}")
        #Compute the train and test errors using the current weights
        tr_error, te_error = ErrorPred(weights, X_train, Y_train, X_test, Y_test, kernelType)
        #Store training and testing errors for plotting
        tr_errors.append(tr_error)
        te_errors.append(te_error)

        #Shuffle the data
        data = np.hstack((X_train, Y_train))
        np.random.shuffle(data)

        batches = np.ceil(X_train.shape[0]/BatchSize)
        batches = int(batches)

        #Iterate through the batches
        for batch in range(batches):
            #Initialize the necessary variables
            gradient = np.zeros((M+1, 1))
            #Get a batch of data
            if batch < batches - 1:
                data_batch = data[batch*BatchSize:(batch+1)*BatchSize]
            else:
                data_batch = data[batch*BatchSize:]

            #Iterate through the data points of obtained batch
            for n in range(len(data_batch)):
                #Obtain kernel representation
                X_tr = []
                x = np.reshape(data_batch[n, 0], (1, 1))
                y = data_batch[n, 1]
                if kernelType=='polynomial':
                    X_tr = np.ones((x.shape[0], 1))
                    curr = np.ones((x.shape[0], 1))

                    for _ in range(M):
                        curr = curr * x
                        X_tr = np.hstack((X_tr, curr))

                if kernelType=='gaussian':
                    X_tr = np.ones((x.shape[0], 1))
                    means = np.linspace(0, 1, M)
                    X, mesh = np.meshgrid(x, means)
                    temp2 = gaussian_kernel(X, mesh)
                    X_tr = np.hstack((X_tr, temp2.T))

                if kernelType=='sigmoidal':
                    X_tr = np.ones((x.shape[0], 1))
                    means = np.linspace(0, 1, M)
                    X, mesh = np.meshgrid(x, means)
                    temp2 = sigmoid_kernel(X, mesh)
                    X_tr = np.hstack((X_tr, temp2.T))

                #Compute the gradient of weight's
                #Compute the running mean of the weights gradients for the batch update
                gradient = gradient + (y - X_tr @ weights) * X_tr.T

            #Update the weights using mean gradient, consider using reasonable stepSize
            weights = weights + stepSize * gradient

    #Plot training and testing error across the epochs
    epoch_nos = range(1, Epochs+1)

    plt.figure(figsize=(15, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epoch_nos, tr_errors, label = "Training Errors")
    plt.legend(loc = 'upper right')
    plt.grid()
    plt.title("Training Errors vs Epoch")

    plt.subplot(1, 2, 2)
    plt.plot(epoch_nos, te_errors, label = "Testing Errors")
    plt.legend(loc = 'upper right')
    plt.grid()
    plt.title("Testing Errors vs Epoch")

    plt.suptitle("Gradient Descent for " + kernelType)
    plt.show()

    #Return the estimated weights
    return weights

def OnlinePred(w_est,X_train,X_test,kernelType): #(estimated weights, training data, testing data, type of the kernel )
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    #Initialize the required variables
    X_tr = []
    X_te = []
    M = len(w_est)-1

    #Obtain kernel representations
    if kernelType=='polynomial':
        X_tr = np.ones((X_train.shape[0], 1))
        curr = np.ones((X_train.shape[0], 1))

        for _ in range(M):
            curr = curr * X_train
            X_tr = np.hstack((X_tr, curr))

        X_te = np.ones((X_test.shape[0], 1))
        curr = np.ones((X_test.shape[0], 1))

        for _ in range(M):
            curr = curr * X_test
            X_te = np.hstack((X_te, curr))

    if kernelType=='gaussian':
        means = np.linspace(0, 1, M)

        X_tr = np.ones((X_train.shape[0], 1))
        temp1, M = np.meshgrid(X_train, means)
        temp = gaussian_kernel(temp1, M)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones((X_test.shape[0], 1))
        temp1, M = np.meshgrid(X_test, means)
        temp = gaussian_kernel(temp1, M)
        X_te = np.hstack((X_te, temp.T))

    if kernelType=='sigmoidal':
        means = np.linspace(0, 1, M)
        
        X_tr = np.ones((X_train.shape[0], 1))
        temp1, M = np.meshgrid(X_train, means)
        temp = sigmoid_kernel(temp1, M)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones((X_test.shape[0], 1))
        temp1, M = np.meshgrid(X_test, means)
        temp = sigmoid_kernel(temp1, M)
        X_te = np.hstack((X_te, temp.T))


    #Estimate training and testing targets
    X_tr = X_tr @ w_est
    X_te = X_te @ w_est

    #Compute and return the training and testing target estimates
    return X_tr, X_te

def OnlinePred_Error(w_est,X_train,Y_train,X_test,Y_test,kernelType): #(estimated weights, training data, training targets, testing data, testing targets, type of the kernel )
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    #Initialize the required variables
    X_tr = []
    X_te = []
    # check this
    X_tr, X_te = OnlinePred(w_est, X_train, X_test, kernelType)
    
    #Compute and return the training and testing errors
    Y_tr_error = np.mean(np.square(X_tr-Y_train))
    Y_te_error = np.mean(np.square(X_te-Y_test))

    return Y_tr_error, Y_te_error

def partStepSizes():
    print("Part Step Sizes")
    # region StepSize
    StepSizes = [1, 0.1, 0.01, 0.001, 0.0001]

    np.random.seed(42)
    input = np.random.uniform(0, 1, (100, 1))
    noise = np.random.normal(0, 0.1, (100, 1))
    output = np.sin(2*(math.pi)*input) + noise

    X_train = np.array([input[i] for i in range(1, 100, 2)])
    X_test = np.array([input[i] for i in range(0, 100, 2)])
    Y_train = np.array([output[i] for i in range(1, 100, 2)])
    Y_test = np.array([output[i] for i in range(0, 100, 2)])

    # print(X_train.shape)
    # print((X_train.shape[0], 1))

    # print(X_test.shape)
    # print((X_test.shape[0], 1))


    M_range=list(range(10))


    for stepSize in StepSizes:
        print(f"Step Size: {stepSize}")
        polynomial_tr_error = []
        polynomial_te_error = []
        gaussian_tr_error = []
        gaussian_te_error = []
        sigmoid_tr_error = []
        sigmoid_te_error = []

        polynomial_train_predictions_list = []
        polynomial_test_predictions_list = []
        gaussian_train_predictions_list = []
        gaussian_test_predictions_list = []
        sigmoid_train_predictions_list = []
        sigmoid_test_predictions_list = []

        Epochs = 1000
        BatchSize = 20
        stepSize = 0.005



        for M in M_range:
            print(f"Model Order: {M}")
            #Fit Mth order polynomial using three kernels i.e {Polynomial,Gaussian,Sigmoid}
            w_polynomial = OnlineTraining(X_train, Y_train, X_test, Y_test, 'polynomial', M, Epochs, BatchSize, stepSize)
            polynomial_train_predictions, polynomial_test_predictions = OnlinePred(w_polynomial, X_train, X_test, 'polynomial')
            polynomial_train_error, polynomial_test_error = OnlinePred_Error(w_polynomial, X_train, Y_train, X_test, Y_test, 'polynomial')
            polynomial_tr_error.append(polynomial_train_error)
            polynomial_te_error.append(polynomial_test_error)
            polynomial_train_predictions_list.append(polynomial_train_predictions)
            polynomial_test_predictions_list.append(polynomial_test_predictions)

            w_gaussian = OnlineTraining(X_train, Y_train, X_test, Y_test, 'gaussian', M, Epochs, BatchSize, stepSize)
            gaussian_train_predictions, gaussian_test_predictions = OnlinePred(w_gaussian, X_train, X_test, 'gaussian')
            gaussian_train_error, gaussian_test_error = OnlinePred_Error(w_gaussian, X_train, Y_train, X_test, Y_test, 'gaussian')
            gaussian_tr_error.append(gaussian_train_error)
            gaussian_te_error.append(gaussian_test_error)
            gaussian_train_predictions_list.append(gaussian_train_predictions)
            gaussian_test_predictions_list.append(gaussian_test_predictions)

            w_sigmoid = OnlineTraining(X_train, Y_train, X_test, Y_test, 'sigmoidal', M, Epochs, BatchSize, stepSize)
            sigmoid_train_predictions, sigmoid_test_predictions = OnlinePred(w_sigmoid, X_train, X_test, 'sigmoidal')
            sigmoid_train_error, sigmoid_test_error = OnlinePred_Error(w_sigmoid, X_train, Y_train, X_test, Y_test, 'sigmoidal')
            sigmoid_tr_error.append(sigmoid_train_error)
            sigmoid_te_error.append(sigmoid_test_error)
            sigmoid_train_predictions_list.append(sigmoid_train_predictions)
            sigmoid_test_predictions_list.append(sigmoid_test_predictions)

            plt.figure(figsize=(20, 5))
            plt.subplot(1, 3, 1)
            plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
            plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
            plt.scatter(X_train, polynomial_train_predictions, color = "red", label = "training data (predicted)")
            plt.scatter(X_test, polynomial_test_predictions, color = "yellow", label = "testing data (predicted)")
            plt.grid()
            plt.legend()
            plt.xlabel("x")
            plt.ylabel("prediction (p(x))")
            plt.xlim(0, 1)
            plt.ylim(-2, 2)
            plt.title(f"polynomial kernel (M = {M})")

            plt.subplot(1, 3, 2)
            plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
            plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
            plt.scatter(X_train, gaussian_train_predictions, color = "red", label = "training data (predicted)")
            plt.scatter(X_test, gaussian_test_predictions, color = "yellow", label = "testing data (predicted)")
            plt.grid()
            plt.legend()
            plt.xlabel("x")
            plt.ylabel("prediction (p(x))")
            plt.xlim(0, 1)
            plt.ylim(-2, 2)
            plt.title(f"gaussian kernel (M = {M})")

            plt.subplot(1, 3, 3)
            plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
            plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
            plt.scatter(X_train, sigmoid_train_predictions, color = "red", label = "training data (predicted)")
            plt.scatter(X_test, sigmoid_test_predictions, color = "yellow", label = "testing data (predicted)")
            plt.grid()
            plt.legend()
            plt.xlabel("x")
            plt.ylabel("prediction (p(x))")
            plt.xlim(0, 1)
            plt.ylim(-2, 2)
            plt.title(f"sigmoid kernel (M = {M})")

            plt.show()

        #Plot the predicted training and testing targets alongside the original targets for various model orders and all three different kernels.


        #Plot training error vs polynomial order and testing error vs polynomial order for all the three different kernels

        # region Plot
        plt.figure(figsize=(20, 5))
        plt.subplot(1, 2, 1)
        plt.plot(M_range, polynomial_tr_error)
        plt.grid()
        plt.xlabel("polynomial order")
        plt.ylabel("error")
        plt.title("polynomial orders vs training errors")

        plt.subplot(1, 2, 2)
        plt.plot(M_range, polynomial_te_error)
        plt.grid()
        plt.xlabel("polynomial order")
        plt.ylabel("error")
        plt.title("polynomial orders vs testing errors")

        plt.suptitle("polynomial kernel errors")
        plt.show()

        plt.figure(figsize=(20, 5))
        plt.subplot(1, 2, 1)
        plt.plot(M_range, gaussian_tr_error)
        plt.grid()
        plt.xlabel("polynomial order")
        plt.ylabel("error")
        plt.title("polynomial orders vs training errors")

        plt.subplot(1, 2, 2)
        plt.plot(M_range, gaussian_te_error)
        plt.grid()
        plt.xlabel("polynomial order")
        plt.ylabel("error")
        plt.title("polynomial orders vs testing errors")

        plt.suptitle("gaussian kernel errors")
        plt.show()

        plt.figure(figsize=(20, 5))
        plt.subplot(1, 2, 1)
        plt.plot(M_range, sigmoid_tr_error)
        plt.grid()
        plt.xlabel("polynomial order")
        plt.ylabel("error")
        plt.title("polynomial orders vs training errors")

        plt.subplot(1, 2, 2)
        plt.plot(M_range, sigmoid_te_error)
        plt.grid()
        plt.xlabel("polynomial order")
        plt.ylabel("error")
        plt.title("polynomial orders vs testing errors")

        plt.suptitle("sigmoid kernel errors")
        plt.show()
        # endregion

    # endregion

partStepSizes()




### part 4 batch size

In [ ]:
def ErrorPred(w_est,X_train,Y_train,X_test,Y_test,kernelType): #(estimated weight, training data, training targets, testing data, testing targets, type of the kernel )
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    X_tr = []
    X_te = []
    M = len(w_est)-1
    if kernelType=='polynomial':
        X_tr = np.ones(X_train.shape)
        temp = np.ones(X_train.shape)

        for _ in range(M):
            temp = temp * X_train
            X_tr = np.hstack((X_tr, temp))

        X_te = np.ones(X_test.shape)
        temp = np.ones(X_test.shape)

        for _ in range(M):
            temp = temp * X_test
            X_te = np.hstack((X_te, temp))

    if kernelType=='gaussian':
        means = np.linspace(0, 1, M)

        X_tr = np.ones(X_train.shape)
        meshx, meshy = np.meshgrid(X_train, means)
        temp = gaussian_kernel(meshx, meshy, 0.1)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones(X_test.shape)
        meshx, meshy = np.meshgrid(X_test, means)
        temp = gaussian_kernel(meshx, meshy, 0.1)
        X_te = np.hstack((X_te, temp.T))

    if kernelType=='sigmoidal':
        means = np.linspace(0, 1, M)
        
        X_tr = np.ones(X_train.shape)
        meshx, meshy = np.meshgrid(X_train, means)
        temp = sigmoid_kernel(meshx, meshy)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones(X_test.shape)
        meshx, meshy = np.meshgrid(X_test, means)
        temp = sigmoid_kernel(meshx, meshy)
        X_te = np.hstack((X_te, temp.T))

    #Estimate training and testing targets
    X_tr = X_tr @ w_est
    X_te = X_te @ w_est

    #Compute and return the training and testing errors
    train_error = np.mean(np.square(X_tr-Y_train))
    test_error = np.mean(np.square(X_te-Y_test))

    return train_error, test_error


def OnlineTraining(X_train,Y_train,X_test, Y_test, kernelType,M,Epochs,BatchSize,stepSize): #(training data, training targets, testing data, testing targets, tupe of the kernel, order of the mode, Number of epochs, Batch size, Step size)
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    #Initialize the weights
    weights = np.zeros((M+1, 1))
    X_train = np.reshape(X_train, (X_train.shape[0], 1))
    Y_train = np.reshape(Y_train, (Y_train.shape[0], 1))
    X_test = np.reshape(X_test, (X_test.shape[0], 1))
    Y_test = np.reshape(Y_test, (Y_test.shape[0], 1))
    #Initialize the necessary variables
    tr_errors = []
    te_errors = []

    #Iterate through epochs

    epochs = range(Epochs)
    for epoch in epochs:
        # print(f"Epoch {epoch+1}")
        #Compute the train and test errors using the current weights
        tr_error, te_error = ErrorPred(weights, X_train, Y_train, X_test, Y_test, kernelType)
        #Store training and testing errors for plotting
        tr_errors.append(tr_error)
        te_errors.append(te_error)

        #Shuffle the data
        data = np.hstack((X_train, Y_train))
        np.random.shuffle(data)

        batches = np.ceil(X_train.shape[0]/BatchSize)
        batches = int(batches)

        #Iterate through the batches
        for batch in range(batches):
            #Initialize the necessary variables
            gradient = np.zeros((M+1, 1))
            #Get a batch of data
            if batch < batches - 1:
                data_batch = data[batch*BatchSize:(batch+1)*BatchSize]
            else:
                data_batch = data[batch*BatchSize:]

            #Iterate through the data points of obtained batch
            for n in range(len(data_batch)):
                #Obtain kernel representation
                X_tr = []
                x = np.reshape(data_batch[n, 0], (1, 1))
                y = data_batch[n, 1]
                if kernelType=='polynomial':
                    X_tr = np.ones((x.shape[0], 1))
                    curr = np.ones((x.shape[0], 1))

                    for _ in range(M):
                        curr = curr * x
                        X_tr = np.hstack((X_tr, curr))

                if kernelType=='gaussian':
                    X_tr = np.ones((x.shape[0], 1))
                    means = np.linspace(0, 1, M)
                    X, mesh = np.meshgrid(x, means)
                    temp2 = gaussian_kernel(X, mesh, 0.1)
                    X_tr = np.hstack((X_tr, temp2.T))

                if kernelType=='sigmoidal':
                    X_tr = np.ones((x.shape[0], 1))
                    means = np.linspace(0, 1, M)
                    X, mesh = np.meshgrid(x, means)
                    temp2 = sigmoid_kernel(X, mesh)
                    X_tr = np.hstack((X_tr, temp2.T))

                #Compute the gradient of weight's
                #Compute the running mean of the weights gradients for the batch update
                gradient = gradient + (y - X_tr @ weights) * X_tr.T

            #Update the weights using mean gradient, consider using reasonable stepSize
            weights = weights + stepSize * gradient

    #Plot training and testing error across the epochs
    epoch_nos = range(1, Epochs+1)

    plt.figure(figsize=(15, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epoch_nos, tr_errors, label = "Training Errors")
    plt.legend(loc = 'upper right')
    plt.grid()
    plt.title("Training Errors vs Epoch")

    plt.subplot(1, 2, 2)
    plt.plot(epoch_nos, te_errors, label = "Testing Errors")
    plt.legend(loc = 'upper right')
    plt.grid()
    plt.title("Testing Errors vs Epoch")

    plt.suptitle("Gradient Descent for " + kernelType)
    plt.show()

    #Return the estimated weights
    return weights

def OnlinePred(w_est,X_train,X_test,kernelType): #(estimated weights, training data, testing data, type of the kernel )
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    #Initialize the required variables
    X_tr = []
    X_te = []
    M = len(w_est)-1

    #Obtain kernel representations
    if kernelType=='polynomial':
        X_tr = np.ones((X_train.shape[0], 1))
        curr = np.ones((X_train.shape[0], 1))

        for _ in range(M):
            curr = curr * X_train
            X_tr = np.hstack((X_tr, curr))

        X_te = np.ones((X_test.shape[0], 1))
        curr = np.ones((X_test.shape[0], 1))

        for _ in range(M):
            curr = curr * X_test
            X_te = np.hstack((X_te, curr))

    if kernelType=='gaussian':
        means = np.linspace(0, 1, M)

        X_tr = np.ones((X_train.shape[0], 1))
        temp1, M = np.meshgrid(X_train, means)
        temp = gaussian_kernel(temp1, M, 0.1)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones((X_test.shape[0], 1))
        temp1, M = np.meshgrid(X_test, means)
        temp = gaussian_kernel(temp1, M, 0.1)
        X_te = np.hstack((X_te, temp.T))

    if kernelType=='sigmoidal':
        means = np.linspace(0, 1, M)
        
        X_tr = np.ones((X_train.shape[0], 1))
        temp1, M = np.meshgrid(X_train, means)
        temp = sigmoid_kernel(temp1, M)
        X_tr = np.hstack((X_tr, temp.T))

        X_te = np.ones((X_test.shape[0], 1))
        temp1, M = np.meshgrid(X_test, means)
        temp = sigmoid_kernel(temp1, M)
        X_te = np.hstack((X_te, temp.T))


    #Estimate training and testing targets
    X_tr = X_tr @ w_est
    X_te = X_te @ w_est

    #Compute and return the training and testing target estimates
    return X_tr, X_te

def OnlinePred_Error(w_est,X_train,Y_train,X_test,Y_test,kernelType): #(estimated weights, training data, training targets, testing data, testing targets, type of the kernel )
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    #Initialize the required variables
    X_tr = []
    X_te = []
    # check this
    X_tr, X_te = OnlinePred(w_est, X_train, X_test, kernelType)
    
    #Compute and return the training and testing errors
    Y_tr_error = np.mean(np.square(X_tr-Y_train))
    Y_te_error = np.mean(np.square(X_te-Y_test))

    return Y_tr_error, Y_te_error


def partBatchSize():
    print("Part Batch Size")
    # region BatchSize
    BatchSizes = [1, 10, 20, 50, 100]

    np.random.seed(42)
    input = np.random.uniform(0, 1, (100, 1))
    noise = np.random.normal(0, 0.1, (100, 1))
    output = np.sin(2*(math.pi)*input) + noise

    X_train = np.array([input[i] for i in range(1, 100, 2)])
    X_test = np.array([input[i] for i in range(0, 100, 2)])
    Y_train = np.array([output[i] for i in range(1, 100, 2)])
    Y_test = np.array([output[i] for i in range(0, 100, 2)])

    # print(X_train.shape)
    # print((X_train.shape[0], 1))

    # print(X_test.shape)
    # print((X_test.shape[0], 1))

    for BatchSize in BatchSizes:
        print(f"Batch Size: {BatchSize}")
        M_range=list(range(10))


        polynomial_tr_error = []
        polynomial_te_error = []
        gaussian_tr_error = []
        gaussian_te_error = []
        sigmoid_tr_error = []
        sigmoid_te_error = []

        polynomial_train_predictions_list = []
        polynomial_test_predictions_list = []
        gaussian_train_predictions_list = []
        gaussian_test_predictions_list = []
        sigmoid_train_predictions_list = []
        sigmoid_test_predictions_list = []

        Epochs = 1000
        # BatchSize = 20
        stepSize = 0.005



        for M in M_range:
            print(f"Model Order: {M}")
            #Fit Mth order polynomial using three kernels i.e {Polynomial,Gaussian,Sigmoid}
            w_polynomial = OnlineTraining(X_train, Y_train, X_test, Y_test, 'polynomial', M, Epochs, BatchSize, stepSize)
            polynomial_train_predictions, polynomial_test_predictions = OnlinePred(w_polynomial, X_train, X_test, 'polynomial')
            polynomial_train_error, polynomial_test_error = OnlinePred_Error(w_polynomial, X_train, Y_train, X_test, Y_test, 'polynomial')
            polynomial_tr_error.append(polynomial_train_error)
            polynomial_te_error.append(polynomial_test_error)
            polynomial_train_predictions_list.append(polynomial_train_predictions)
            polynomial_test_predictions_list.append(polynomial_test_predictions)
            
            
            w_gaussian = OnlineTraining(X_train, Y_train, X_test, Y_test, 'gaussian', M, Epochs, BatchSize, stepSize)
            gaussian_train_predictions, gaussian_test_predictions = OnlinePred(w_gaussian, X_train, X_test, 'gaussian')
            gaussian_train_error, gaussian_test_error = OnlinePred_Error(w_gaussian, X_train, Y_train, X_test, Y_test, 'gaussian')
            gaussian_tr_error.append(gaussian_train_error)
            gaussian_te_error.append(gaussian_test_error)
            gaussian_train_predictions_list.append(gaussian_train_predictions)
            gaussian_test_predictions_list.append(gaussian_test_predictions)
            
            
            w_sigmoid = OnlineTraining(X_train, Y_train, X_test, Y_test, 'sigmoidal', M, Epochs, BatchSize, stepSize)
            sigmoid_train_predictions, sigmoid_test_predictions = OnlinePred(w_sigmoid, X_train, X_test, 'sigmoidal')
            sigmoid_train_error, sigmoid_test_error = OnlinePred_Error(w_sigmoid, X_train, Y_train, X_test, Y_test, 'sigmoidal')
            sigmoid_tr_error.append(sigmoid_train_error)
            sigmoid_te_error.append(sigmoid_test_error)
            sigmoid_train_predictions_list.append(sigmoid_train_predictions)
            sigmoid_test_predictions_list.append(sigmoid_test_predictions)

            plt.figure(figsize=(20, 5))
            plt.subplot(1, 3, 1)
            plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
            plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
            plt.scatter(X_train, polynomial_train_predictions, color = "red", label = "training data (predicted)")
            plt.scatter(X_test, polynomial_test_predictions, color = "yellow", label = "testing data (predicted)")
            plt.grid()
            plt.legend()
            plt.xlabel("x")
            plt.ylabel("prediction (p(x))")
            plt.xlim(0, 1)
            plt.ylim(-2, 2)
            plt.title(f"polynomial kernel (M = {M})")

            plt.subplot(1, 3, 2)
            plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
            plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
            plt.scatter(X_train, gaussian_train_predictions, color = "red", label = "training data (predicted)")
            plt.scatter(X_test, gaussian_test_predictions, color = "yellow", label = "testing data (predicted)")
            plt.grid()
            plt.legend()
            plt.xlabel("x")
            plt.ylabel("prediction (p(x))")
            plt.xlim(0, 1)
            plt.ylim(-2, 2)
            plt.title(f"gaussian kernel (M = {M})")

            plt.subplot(1, 3, 3)
            plt.scatter(X_train, Y_train, color = "blue", label = "training data (true)")
            plt.scatter(X_test, Y_test, color = "cyan", label = "testing data (true)")
            plt.scatter(X_train, sigmoid_train_predictions, color = "red", label = "training data (predicted)")
            plt.scatter(X_test, sigmoid_test_predictions, color = "yellow", label = "testing data (predicted)")
            plt.grid()
            plt.legend()
            plt.xlabel("x")
            plt.ylabel("prediction (p(x))")
            plt.xlim(0, 1)
            plt.ylim(-2, 2)
            plt.title(f"sigmoid kernel (M = {M})")

            plt.show()

        #Plot the predicted training and testing targets alongside the original targets for various model orders and all three different kernels.


        #Plot training error vs polynomial order and testing error vs polynomial order for all the three different kernels

        # region Plot
        plt.figure(figsize=(20, 5))
        plt.subplot(1, 2, 1)
        plt.plot(M_range, polynomial_tr_error)
        plt.grid()
        plt.xlabel("polynomial order")
        plt.ylabel("error")
        plt.title("polynomial order vs training errors")

        plt.subplot(1, 2, 2)
        plt.plot(M_range, polynomial_te_error)
        plt.grid()
        plt.xlabel("polynomial order")
        plt.ylabel("error")
        plt.title("polynomial order vs testing errors")

        plt.suptitle("polynomial kernel errors")
        plt.show()

        plt.figure(figsize=(20, 5))
        plt.subplot(1, 2, 1)
        plt.plot(M_range, gaussian_tr_error)
        plt.grid()
        plt.xlabel("polynomial order")
        plt.ylabel("error")
        plt.title("polynomial order vs training errors")

        plt.subplot(1, 2, 2)
        plt.plot(M_range, gaussian_te_error)
        plt.grid()
        plt.xlabel("polynomial order")
        plt.ylabel("error")
        plt.title("polynomial order vs testing errors")

        plt.suptitle("gaussian kernel errors")
        plt.show()

        plt.figure(figsize=(20, 5))
        plt.subplot(1, 2, 1)
        plt.plot(M_range, sigmoid_tr_error)
        plt.grid()
        plt.xlabel("polynomial order")
        plt.ylabel("error")
        plt.title("polynomial order vs training errors")

        plt.subplot(1, 2, 2)
        plt.plot(M_range, sigmoid_te_error)
        plt.grid()
        plt.xlabel("polynomial order")
        plt.ylabel("error")
        plt.title("polynomial order vs testing errors")

        plt.suptitle("sigmoid kernel errors")
        plt.show()
        # endregion

    # endregion

partBatchSize()



<b> Report your observations </b>

1. In part a, all kernels converged equally well. Sometimes the solution was not optimum, due to error surface different in each epoch of training and thus weight oscillations.

2. In part b, gaussian and sigmoid kernels performed well, but polynomial kernel failed due to numerical overflow.

3. In part c, for small step size the convergence is slower but solution is optimum. For large step size, the convergence is faster but the solution is not optimum (oscillationg).

4. Small batch size shows more oscillations but error is lower due to escaping local minima, while for higher batch size the convergence is faster but requires small stepsize to give optimum solution



# PART 5

In [ ]:
#Understanding the bias-variance trade-off
########################################
#Generate 100 data sets of noisy sinusoidal data
########################################

def KernelRegressionFit(X_train,Y_train,kernelType,M,lamda): #(training data, training targets, type of kernel, regularization coefficient)
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    #print(X_train.shape)
    #X_train = np.reshape(X_train, (1,-1))
    #Y_train = np.reshape(Y_train, (1,-1))
    X = []

    if kernelType=='gaussian':
        #Use Gaussian kernel to transform the data
        X = np.ones((X_train.shape[0], 1))
        means = np.linspace(-0, 1, M)
        ins, mean_list = np.meshgrid(X_train, means)
        temp2 = gaussian_kernel(ins, mean_list)
        X = np.hstack((X, temp2.T))

    #Estimate weights using Pseudo iverse solution
    w_opt = np.linalg.inv(X.T @ X + lamda*np.identity(X.shape[1])) @ (X.T) @ Y_train

    #Return the estimated weights
    return w_opt


def KernelRegressionPred_Error(w_est,X_train,kernelType):
    #kernelType : {Polynomial,Gaussian,Sigmoid}
    X_tr = []
    X_te = []
    #X_train = np.reshape(X_train, (1,-1))
    #Y_train = np.reshape(Y_train, (1,-1))
    #X_test = np.reshape(X_test, (1,-1))
    #Y_test = np.reshape(Y_test, (1,-1))
    M = len(w_est)-1

    if kernelType=='gaussian':
        #Use Gaussian kernel to transform the data
        means = np.linspace(-0, 1, M)

        X_tr = np.ones(X_train.shape)
        ins, mean_list = np.meshgrid(X_train, means)
        temp = gaussian_kernel(ins, mean_list)
        X_tr = np.hstack((X_tr, temp.T))

        # X_te = np.ones(X_test.shape)
        # ins, mean_list = np.meshgrid(X_test, means)
        # temp = gaussian_kernel(ins, mean_list, 1)
        # X_te = np.hstack((X_te, temp.T))


    #Estimate training and testing targets
    train_predictions = X_tr @ w_est

    #Compute and return the training and testing errors

    return train_predictions


########################################
#Use regularized least squares to estimate w
########################################
lamdas = [0.00000000001, 1, 1000]
w_opt_lambda_low = []
w_opt_lambda_middle = []
w_opt_lambda_high = []

for _ in range(100):
    input = np.random.uniform(0, 1, (25, 1))
    noise = np.random.normal(0, 0.1, (25, 1))
    output = (np.sin(2*(math.pi)*input) + noise).reshape(25,1)

    w_opt = KernelRegressionFit(input, output, 'gaussian', 24, lamdas[0])
    w_opt_lambda_low.append(w_opt)

    w_opt = KernelRegressionFit(input, output, 'gaussian', 24, lamdas[1])
    w_opt_lambda_middle.append(w_opt)

    w_opt = KernelRegressionFit(input, output, 'gaussian', 24, lamdas[2])
    w_opt_lambda_high.append(w_opt)




########################################
#Illustrate the concept of Bias-Variance trade off
########################################
#1. Chose three different regularization coefficeints (low,middle and high)
#2. For every regularization coefficient, produce two plots: one displaying 100 estimated curves, and the other showing the mean of the estimated curves alongside the original function.
#3. For three regularization coefficients, you should have a total of six plots, meaning two plots for each regularization.
#4. Using the six plots above, describe the bias-variance trade-off.

X_train = np.linspace(0, 1, 100).reshape(100,1)
y = np.sin(2*(math.pi)*X_train)

y_lambda_low = []
y_lambda_middle = []
y_lambda_high = []


# region PlotLowLambda

plt.figure(figsize=(15, 5))
plt.suptitle('predictions (low lambda)')

plt.subplot(1, 2, 1)
plt.grid()
plt.xlabel('x')
plt.ylabel('f(x)')
plt.xlim(0, 1)
plt.ylim(-1.5, 1.5)
for _ in range(100):
    y = KernelRegressionPred_Error(w_opt_lambda_low[_],X_train,'gaussian')
    plt.plot(X_train, y)
    y_lambda_low.append(y)

plt.subplot(1, 2, 2)
plt.xlim(0, 1)
plt.ylim(-1.5, 1.5)
plt.plot(X_train, np.mean(y_lambda_low, axis=0), label = 'avg prediction')
plt.plot(X_train, y, label = 'function (true)')
plt.grid()
plt.legend()
plt.xlabel('x')
plt.ylabel('f(x)')

plt.show()

# endregion


# region PlotMiddleLambda

plt.figure(figsize=(15, 5))
plt.suptitle('predictions (middle lambda)')

plt.subplot(1, 2, 1)
plt.grid()
plt.xlabel('x')
plt.ylabel('f(x)')
plt.xlim(0, 1)
plt.ylim(-1.5, 1.5)
for _ in range(100):
    y = KernelRegressionPred_Error(w_opt_lambda_middle[_],X_train,'gaussian')
    plt.plot(X_train, y)
    y_lambda_middle.append(y)

plt.subplot(1, 2, 2)
plt.xlim(0, 1)
plt.ylim(-1.5, 1.5)
plt.plot(X_train, np.mean(y_lambda_middle, axis=0), label = 'avg prediction')
plt.plot(X_train, y, label = 'function (true)')
plt.grid()
plt.legend()
plt.xlabel('x')
plt.ylabel('f(x)')

plt.show()

# endregion


# region PlotHighLambda
plt.figure(figsize=(15, 5))
plt.suptitle('predictions (hugh lambda)')

plt.subplot(1, 2, 1)
plt.grid()
plt.xlabel('x')
plt.ylabel('f(x)')
plt.xlim(0, 1)
plt.ylim(-1.5, 1.5)
for _ in range(100):
    y = KernelRegressionPred_Error(w_opt_lambda_high[_],X_train,'gaussian')
    plt.plot(X_train, y)
    y_lambda_high.append(y)

plt.subplot(1, 2, 2)
plt.xlim(0, 1)
plt.ylim(-1.5, 1.5)
plt.plot(X_train, np.mean(y_lambda_high, axis=0), label = 'avg prediction')
plt.plot(X_train, y, label = 'function (true)')
plt.grid()
plt.legend()
plt.xlabel('x')
plt.ylabel('f(x)')

plt.show()

# endregion


<b> Report your observations </b>

1. The top graph predictions have high variance as can be seen from x=0 and x=1, and low bias as the mean is close to the original function.

2. The middle graph predictions have medium variance and bias, as the mean is close to the original function, and the variance is not too high.

3. The lower graph predictions have low variance and high bias, as the mean is far from the original function, and the variance is low as predictions are close to each other



# PART 6

In [ ]:
np.random.seed(42)
M = 20
alpha = 1
beta = 25
test_variance = 0.2

x_train = np.random.uniform(0, 1, (100, 1))
noise = np.random.normal(0, 0.1, (100, 1))
y_train = np.sin(2*(math.pi)*x_train) + noise 

shuffled_indices = np.random.permutation(len(x_train))
x_train = x_train[shuffled_indices]
y_train = y_train[shuffled_indices]


########################################
#Update the statistics of posterior density
########################################
#Initialie the parameters for standard normal prior


current_mean = np.zeros(M)
current_variance = np.identity(M)
mean_list = np.linspace(0, 1, M)

def gaussian_kernel(X, mean, variance = 0.1):
    return np.exp(-1 * np.square(X - mean) / (2 * variance**2))

#Iterate through the data points and update the stats of posterior density


def posterior_distribution_update(x_n, y_n, means, variances,prior_mean, prior_variance, precision):
    y_n = y_n.reshape(-1, 1)
    prior_mean = prior_mean.reshape(-1, 1)
    kernel = gaussian_kernel(x_n.reshape(-1, 1), means.reshape(1,-1), variances)
    updated_variance = np.linalg.inv(np.linalg.inv(prior_variance) + precision * kernel.T@kernel)
    updated_mean = (updated_variance @ np.linalg.inv(prior_variance)) @ prior_mean + updated_variance @ (precision * (kernel.T) @ y_n)
    return updated_mean, updated_variance

########################################
#Sample weight vector from posterior distribution. Estimate the curve, repeat the procedure for 100 times and get the avg fit
########################################

for _ in range(x_train.shape[0]):
    current_mean, current_variance = posterior_distribution_update(x_train[_], y_train[_], mean_list, test_variance, current_mean, current_variance, beta)

sample = (np.random.multivariate_normal(current_mean.ravel(), current_variance)).reshape(-1, 1)
x_kernel = gaussian_kernel(x_train, mean_list, test_variance)

y_pred = x_kernel @ sample

x_coords = np.linspace(0, 1, 1000).reshape(1000, 1) 
y_coords = np.sin(2 * np.pi * x_coords)

plt.scatter(x_train, y_pred, color='pink')
plt.plot(x_coords, y_coords, label="function (true)", color="red")
plt.xlabel("X")
plt.ylabel("Y")
plt.legend()
plt.grid(True)
plt.show()

curve_fits = np.zeros((x_train.shape[0], 100))
for _ in range(100):
    sample = np.random.multivariate_normal(current_mean.ravel(), current_variance).reshape(-1, 1)
    x_kernel = gaussian_kernel(x_train, mean_list, test_variance)
    y_pred = x_kernel @ sample  
    curve_fits[:, _] = y_pred.ravel() 


y_avg = np.mean(curve_fits, axis=1)


x_coords = np.linspace(0, 1, 1000).reshape(1000, 1)
y_coords = np.sin(2 * np.pi * x_coords)
index_sorted = np.argsort(x_train)

x_train_sorted = x_train[index_sorted].reshape(-1, 1)
y_train_sorted = y_train[index_sorted].reshape(-1, 1)
y_avg_sorted = y_avg[index_sorted].reshape(-1, 1)
plt.plot(x_train_sorted, y_avg_sorted, label='avg curve', color='blue')
plt.plot(x_coords, y_coords, label='function (true)', color='red')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.title("true function vs avg")
plt.grid(True)
plt.show()

########################################
#Predictive distribution analysis
########################################
#Predictive distribution analysis through sampling
#Iterate through data points and sample weight vectors when partial data points are seen, and plot the curves



sample_sizes = [1, 2, 10, 50, 100]
for sample_size in sample_sizes:
    current_mean = np.zeros(M)
    current_variance = np.identity(M)
    mean_list = np.linspace(0, 1, M)
    x_subset = x_train[:sample_size]
    y_subset = y_train[:sample_size]
    
    for _ in range(sample_size):
        current_mean, current_variance = posterior_distribution_update(x_subset[_], y_subset[_], mean_list, test_variance, current_mean, current_variance, beta)
    
    predicted_means = []
    predicted_variances = []


    for _ in range(len(x_train)): 
        x_point = x_train[_].reshape(-1, 1)
        kernel = gaussian_kernel(x_point, mean_list, test_variance)
        
        pred_mean = float(kernel @ current_mean)
        pred_var = float(beta ** -1 + kernel @ current_variance @ kernel.T)
        
        predicted_means.append(pred_mean)
        predicted_variances.append(pred_var)
    
    predicted_means = np.array(predicted_means)
    predicted_variances = np.array(predicted_variances)
    
    index_sorted = np.argsort(x_train.flatten())
    x_sorted = x_train.flatten()[index_sorted]
    means_sorted = predicted_means[index_sorted]
    vars_sorted = predicted_variances[index_sorted]
    std_deviation_sorted = np.sqrt(vars_sorted)
    
    plt.figure(figsize=(8, 5))
    plt.fill_between(
        x_sorted,
        means_sorted - 2 * std_deviation_sorted,
        means_sorted + 2 * std_deviation_sorted,
        color='pink', label='confidence interval (±2 std deviation)'
    )
    plt.plot(x_coords, y_coords, label='function (true)')
    plt.plot(x_sorted, means_sorted, label='function (predicted)')
    plt.scatter(x_subset, y_subset, label='train data', color='black')
    plt.title(f'confidence intervals (sample_size={sample_size})')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.show()

<b> Report your observations </b>

1. MAP estimate is equivalent to LSE with gaussian priors. As updates approach 100, the predictions become increasingly accurate

2. Analysis shows that as sample size increases, the sampled curves approach the true curve and variances decrease.

3. Confidence in value prediction near already sampled points is higher than near non sampled intervals



In [ ]:
np.random.seed(42)
M = 20
alpha = 1
beta = 25
test_variance = 0.2

x_train = np.random.uniform(0, 1, (100, 1))
noise = np.random.normal(0, 0.1, (100, 1))
y_train = np.sin(2*(math.pi)*x_train) + noise 

shuffled_indices = np.random.permutation(len(x_train))
x_train = x_train[shuffled_indices]
y_train = y_train[shuffled_indices]


########################################
#Update the statistics of posterior density
########################################
#Initialie the parameters for standard normal prior


current_mean = np.zeros(M)
current_variance = np.identity(M)
mean_list = np.linspace(0, 1, M)

def gaussian_kernel(X, mean, variance = 0.1):
    return np.exp(-1 * np.square(X - mean) / (2 * variance**2))

#Iterate through the data points and update the stats of posterior density


def posterior_distribution_update(x_n, y_n, means, variances,prior_mean, prior_variance, precision):
    y_n = y_n.reshape(-1, 1)
    prior_mean = prior_mean.reshape(-1, 1)
    kernel = gaussian_kernel(x_n.reshape(-1, 1), means.reshape(1,-1), variances)
    updated_variance = np.linalg.inv(np.linalg.inv(prior_variance) + precision * kernel.T@kernel)
    updated_mean = (updated_variance @ np.linalg.inv(prior_variance)) @ prior_mean + updated_variance @ (precision * (kernel.T) @ y_n)
    return updated_mean, updated_variance

########################################
#Sample weight vector from posterior distribution. Estimate the curve, repeat the procedure for 100 times and get the avg fit
########################################

for _ in range(x_train.shape[0]):
    current_mean, current_variance = posterior_distribution_update(x_train[_], y_train[_], mean_list, test_variance, current_mean, current_variance, beta)

sample = (np.random.multivariate_normal(current_mean.ravel(), current_variance)).reshape(-1, 1)
x_kernel = gaussian_kernel(x_train, mean_list, test_variance)

y_pred = x_kernel @ sample

x_coords = np.linspace(0, 1, 1000).reshape(1000, 1) 
y_coords = np.sin(2 * np.pi * x_coords)

plt.scatter(x_train, y_pred, color='pink')
plt.plot(x_coords, y_coords, label="function (true)", color="red")
plt.xlabel("X")
plt.ylabel("Y")
plt.legend()
plt.grid(True)
plt.show()

curve_fits = np.zeros((x_train.shape[0], 100))
for _ in range(100):
    sample = np.random.multivariate_normal(current_mean.ravel(), current_variance).reshape(-1, 1)
    x_kernel = gaussian_kernel(x_train, mean_list, test_variance)
    y_pred = x_kernel @ sample  
    curve_fits[:, _] = y_pred.ravel() 


y_avg = np.mean(curve_fits, axis=1)


x_coords = np.linspace(0, 1, 1000).reshape(1000, 1)
y_coords = np.sin(2 * np.pi * x_coords)
index_sorted = np.argsort(x_train)

x_train_sorted = x_train[index_sorted].reshape(-1, 1)
y_train_sorted = y_train[index_sorted].reshape(-1, 1)
y_avg_sorted = y_avg[index_sorted].reshape(-1, 1)
plt.plot(x_train_sorted, y_avg_sorted, label='avg curve', color='blue')
plt.plot(x_coords, y_coords, label='function (true)', color='red')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.title("true function vs avg")
plt.grid(True)
plt.show()

########################################
#Predictive distribution analysis
########################################
#Predictive distribution analysis through sampling
#Iterate through data points and sample weight vectors when partial data points are seen, and plot the curves



sample_sizes = [1, 2, 10, 50, 100]
for sample_size in sample_sizes:
    current_mean = np.zeros(M)
    current_variance = np.identity(M)
    mean_list = np.linspace(0, 1, M)
    x_subset = x_train[:sample_size]
    y_subset = y_train[:sample_size]
    
    for _ in range(sample_size):
        current_mean, current_variance = posterior_distribution_update(x_subset[_], y_subset[_], mean_list, test_variance, current_mean, current_variance, beta)
    
    predicted_means = []
    predicted_variances = []


    for _ in range(len(x_train)): 
        x_point = x_train[_].reshape(-1, 1)
        kernel = gaussian_kernel(x_point, mean_list, test_variance)
        
        pred_mean = float(kernel @ current_mean)
        pred_var = float(beta ** -1 + kernel @ current_variance @ kernel.T)
        
        predicted_means.append(pred_mean)
        predicted_variances.append(pred_var)
    
    predicted_means = np.array(predicted_means)
    predicted_variances = np.array(predicted_variances)
    
    index_sorted = np.argsort(x_train.flatten())
    x_sorted = x_train.flatten()[index_sorted]
    means_sorted = predicted_means[index_sorted]
    vars_sorted = predicted_variances[index_sorted]
    std_deviation_sorted = np.sqrt(vars_sorted)
    
    plt.figure(figsize=(8, 5))
    plt.fill_between(
        x_sorted,
        means_sorted - 2 * std_deviation_sorted,
        means_sorted + 2 * std_deviation_sorted,
        color='pink', label='confidence interval (±2 std deviation)'
    )
    plt.plot(x_coords, y_coords, label='function (true)')
    plt.plot(x_sorted, means_sorted, label='function (predicted)')
    plt.scatter(x_subset, y_subset, label='train data', color='black')
    plt.title(f'confidence intervals (sample_size={sample_size})')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.show()